# 02 — Sửa lỗi dtype, rồi ĐO những gì notebook 01 chưa đo được

**Notebook 01 đã trả lời xong**: weights cache được (24.9 GB, không tải lại nữa), model load được
(6.41 GB VRAM ở bf16+fp8, 105.7 s), `kernel_backend="fused"` chạy, `msa_encoder` có thật,
thứ tự token == thứ tự residue (`is_identity=true` cả 5 lần) → **rủi ro permutation đã đóng**.

**Nhưng mọi lần fold đều chết cùng một lỗi:**

```
RuntimeError: mat1 and mat2 must have the same dtype, but got Float and BFloat16
```

## Nguyên nhân gốc — đã tìm ra bằng cách đọc source, không phải đoán

`esm/models/esmfold2/model.py`, `EsmFold2Model.forward`:

| dòng | code | nghĩa |
|---|---|---|
| 973 | `with torch.amp.autocast("cuda", enabled=use_amp, dtype=torch.bfloat16):` | **mở** autocast |
| 974–1157 | inputs_embedder, pair init, LM encoder, MSA encoder, recycle loop, parcae | toàn bộ **trunk** chạy trong autocast |
| 1159 | `z = z.float()` | pair rep được **upcast lên fp32**, vẫn trong autocast |
| **1160** | *(hết block)* | **autocast ĐÓNG ở đây** |
| 1181 | `distogram_logits = self.distogram_head(z + z.transpose(-2, -3))` | fp32 × weight → **NGOÀI autocast** |
| 1185 | `self.structure_head.sample(z_trunk=z, ...)` | diffusion decoder, fp32 |
| 1239 | `self.confidence_head(..., z=z.detach().float(), ...)` | head sinh PAE/pLDDT, fp32 |

Ba head sau trunk được **thiết kế để chạy fp32**. `DiffusionConditioning.forward`
(`layers.py:1520-1534`) còn tự cưỡng chế: `z_trunk.to(torch.float32)`,
`s_inputs_eff.to(torch.float32)`. `ConfidenceHead` (`model.py:221`) làm
`pair.add_(pair_delta.float())` rồi `self.pae_head(self.pae_ln(pair))` trên fp32.

Nhưng `from_pretrained(..., dtype=torch.bfloat16)` gọi `model.to(torch.bfloat16)`
(`hub.py:194`) — cast **toàn bộ** model, kể cả ba head đó. Linear đầu tiên gặp phải là
`self.distogram_head` (một `nn.Linear(256, 64)` trần) ở dòng 1181:
**activation fp32 × weight bf16 → đúng cái lỗi ta thấy.**

**Mảnh ghép cuối, chứng minh fp32 là cấu hình *đầy đủ* chứ không phải bản vá:**
`InputsEmbedder.forward` (`layers.py:2136`) trả về
`torch.cat([a, aatype, profile, deletion_mean], dim=-1)` — trong đó `a` là bf16 (ra từ
atom encoder trong autocast) nhưng `aatype`/`profile`/`deletion_mean` được cưỡng chế
`.float()` ở `model.py:936/976/977`. `torch.cat` **nâng kiểu** → `x_inputs` là **fp32**.
Và `x_inputs` chính là thứ đi vào `structure_head.sample(s_inputs=...)` (1187) và
`confidence_head(s_inputs=...)` (1240). Nghĩa là **cả hai đầu vào của ba head — `z` và
`x_inputs` — đều fp32 một cách có chủ ý.** Load fp32 làm mọi thứ khớp nhau; không còn
chỗ nào lệch dtype.

→ Suy ra ba điều:

1. **Diffusion decoder chưa từng chạy.** Lỗi xảy ra ở `distogram_head`, ngay sau trunk.
   Mọi số VRAM/thời gian của notebook 01 là **sàn dưới của sàn dưới**.
2. **`esmc_precision` không liên quan.** Giả thuyết "đổi sang `bf16`" và
   "`bf16` + `_offload_esmc`" **sẽ chết y nguyên** — không tiêu compute để thử.
3. **Cách sửa đúng là `dtype=None` (fp32).** Trunk *vẫn* chạy bf16 vì autocast ở dòng 973
   lo việc đó, `FoldingTrunk.forward` (`layers.py:2834-2840`) còn tự cast pair sang bf16
   khi bật fused kernel, và các fused kernel cast weight tại call site
   (`_to_fused_kernel_dtype`, `layers.py:2625`). Giá phải trả: weight trunk fp32 thay vì
   bf16 = **+0.53 GB**. Đổi lại đúng cấu hình mà tác giả thiết kế.

| Cấu hình | ESMC | Trunk+head | Tổng weights | Dự đoán |
|---|---|---|---|---|
| `dtype=bfloat16, fp8` (notebook 01) | 5.9 GB | 0.53 GB | 6.41 GB **đo được** | ❌ chết ở `distogram_head` |
| **`dtype=None, fp8`** ← dùng cái này | 5.9 GB | 1.07 GB | **≈ 6.95 GB** | ✅ đường tác giả thiết kế |
| `dtype=None, bf16` | 12.7 GB | 1.07 GB | ≈ 13.8 GB | ✅ nhưng tốn gấp đôi, vô ích |

## Notebook này đo gì

| # | Đo | Vì sao quan trọng |
|---|---|---|
| 1 | **Chứng minh nguyên nhân** bằng micro-repro 2 dòng (0 giây GPU) | khỏi phải đoán |
| 2 | Fold tối thiểu (80 aa, 1 loop, 4 step) qua từng cấu hình đến khi `result.pae` về | chốt dtype **rẻ nhất có thể** |
| 3 | **ipSAE binder 7S5B thật vs bản xáo trộn** | **kết quả quan trọng nhất cả notebook.** Thước đo không phân biệt được binder thật khỏi sequence xáo thì mọi thứ phía sau vô nghĩa |
| 4 | **Tính tiền định**: fold y hệt 2 lần `seed=0` → PAE có giống từng bit? | cửa sàng ghi "1 seed / 1 sample". Nếu `lm_dropout=0.3` làm nó ngẫu nhiên thì câu đó vô nghĩa |
| 5 | VRAM + thời gian **thật** ở cấu hình cửa sàng (20 loop / 200 step) | quyết định toàn bộ kiến trúc notebook 03 |
| 6 | **Trần residue** 280 → 420 → 540 → 640 → 775 aa | TL1A fold trimer được không |
| 7 | **monomer vs trimer**: cùng DcR3 (binder TL1A thật, PDB 3K51) đấu 1 protomer vs 3 | DcR3 bám **rãnh giữa hai protomer** → nếu hai số lệch nhau, đây là rủi ro lớn nhất của nhánh TL1A và phải hỏi organizer |
| 8 | **ipSAE có MSA vs không MSA** | MSA thật depth 921 đã nhúng sẵn trong notebook → không phụ thuộc mạng nữa |

**GPU**: L4 (≥ 20 GB). **Thời gian**: ~35–50 phút. **Ước**: 3–5 CU (trần cứng 75 phút = 6 CU).
**Không tải lại weights** — cell 3 chỉ verify cache rồi dừng cứng nếu thiếu.

## Cell 1 — TIỀN KIỂM

Kiểm GPU / VRAM / Drive / cache weights **trước** khi làm gì tốn thời gian.
Thiếu điều kiện → `raise SystemExit` kèm lời nhắc cụ thể.

In [ ]:
# ============================== CELL 1: PRE-FLIGHT ==============================
import os, sys, time, json, shutil, subprocess, platform

# Phải set TRƯỚC khi import torch. Giảm phân mảnh — quan trọng cho probe trần residue.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# ---------------- knobs (đổi ở đây, đừng sửa rải rác) ----------------
HARD_CAP_MIN        = 75       # trần cứng: vượt là dừng (75 phút ≈ 6.0 CU)
CU_PER_HOUR         = 4.8      # L4 Colab Pro

# dtype: None = fp32 (ĐÚNG — xem cell 0). "bfloat16" = cấu hình đã chết của notebook 01.
MODEL_DTYPE         = None
ESMC_PRECISION      = "fp8"

# cấu hình cửa sàng (giả định; hai dòng dưới là hai câu PHẢI hỏi organizer)
GATE_NUM_LOOPS      = 20       # config.json: num_loops = 20  (khớp)
GATE_SAMPLING_STEPS = 200      # fold() default = 200  NHƯNG config.json inference_num_steps = 14  <-- LỆCH
GATE_LM_DROPOUT     = 0.3      # fold() default = 0.3 ("paper folding-eval value")
                               # NHƯNG config.json lm_encoder.lm_dropout = 0.25             <-- LỆCH
MSA_MAX_DEPTH       = 1024
MSA_COLUMN_MASK_RATE = 0.1
PAE_CUTOFF, DIST_CUTOFF = 10, 10

# probe trần residue: rẻ, chỉ cần biết có OOM hay không
PROBE_NUM_LOOPS     = 4
PROBE_SAMPLING_STEPS = 8

# monomer-vs-trimer: hai nhánh dùng Y HỆT tham số, nếu không thì delta vô nghĩa
MT_NUM_LOOPS        = 10
MT_SAMPLING_STEPS   = 50

# bật/tắt từng nhóm nếu cần tiết kiệm (mặc định bật hết)
DO_DETERMINISM      = True
DO_MSA_DELTA        = True
DO_CEILING_PROBE    = True
DO_MONO_TRI         = True

STAGE_TO_LOCAL      = True     # load weights từ /content thay vì Drive FUSE
UA = "proteinAI-changjiangcup/0.2 (notebook 02)"

T0 = time.time()
def elapsed_min(): return (time.time() - T0) / 60.0
def cu_burned():   return elapsed_min() / 60.0 * CU_PER_HOUR

def budget(step=""):
    m, cu = elapsed_min(), cu_burned()
    bar = "!!!" if m > 0.8 * HARD_CAP_MIN else ("!" if m > 0.5 * HARD_CAP_MIN else "")
    print(f"[NGÂN SÁCH]{bar} {m:6.1f} phút · ước {cu:4.2f} CU "
          f"· trần {HARD_CAP_MIN} phút ({HARD_CAP_MIN/60*CU_PER_HOUR:.1f} CU) {step}", flush=True)
    if m > HARD_CAP_MIN:
        try: save_partial()
        except Exception: pass
        raise SystemExit(
            f"DỪNG CỨNG: đã chạy {m:.0f} phút (> HARD_CAP_MIN={HARD_CAP_MIN}), ước {cu:.1f} CU. "
            f"Kết quả từng phần ĐÃ ghi ra out/02_partial.json trên Drive — gửi file đó cho Claude.")

def have_budget(need_min, what):
    """True nếu còn đủ thời gian cho bước tới. In lý do nếu bỏ qua."""
    left = HARD_CAP_MIN - elapsed_min()
    if left < need_min:
        print(f"BỎ QUA {what}: còn {left:.0f} phút, bước này cần ~{need_min} phút.")
        return False
    return True

ENV = {"python": platform.python_version(), "platform": platform.platform()}

try:
    import torch
except Exception as e:
    raise SystemExit(f"DỪNG: không import được torch ({e}).")

if not torch.cuda.is_available():
    raise SystemExit("DỪNG: KHÔNG CÓ GPU. Runtime > Change runtime type > GPU (L4), rồi Run all lại.")

gpu_name = torch.cuda.get_device_name(0)
cc = torch.cuda.get_device_capability(0)
free_b, total_b = torch.cuda.mem_get_info(0)
vram_total_gb, vram_free_gb = total_b / 1024**3, free_b / 1024**3
ENV.update(torch_version=torch.__version__, gpu=gpu_name, compute_capability=list(cc),
           vram_total_gb=round(vram_total_gb, 2), vram_free_gb=round(vram_free_gb, 2),
           alloc_conf=os.environ.get("PYTORCH_CUDA_ALLOC_CONF"))
print(f"GPU    : {gpu_name}  (cc {cc[0]}.{cc[1]})")
print(f"VRAM   : {vram_total_gb:.2f} GB tổng · {vram_free_gb:.2f} GB trống")
print(f"torch  : {torch.__version__}")

if vram_total_gb < 20.0:
    raise SystemExit(
        f"DỪNG: GPU '{gpu_name}' chỉ {vram_total_gb:.1f} GB VRAM, cần ≥ 20 GB "
        f"(weights fp32-trunk + ESMC fp8 ≈ 7 GB, phần còn lại cho activation + diffusion). "
        f"Nếu đang T4 16 GB: đổi sang L4.")
if cc[0] * 10 + cc[1] < 89:
    print(f"CẢNH BÁO: cc {cc[0]}.{cc[1]} < 8.9 → không có fp8 GEMM. Weight vẫn fp8 (tiết kiệm VRAM) "
          f"nhưng mỗi matmul phải upcast → chậm hơn.")

try:
    from google.colab import drive  # noqa
    drive.mount("/content/drive")
except Exception as e:
    raise SystemExit(f"DỪNG: không mount được Drive ({e}). Notebook này bắt buộc cần cache weights trên Drive.")

ROOT   = "/content/drive/MyDrive/proteinAI"
D_W    = f"{ROOT}/weights/ESMFold2"
D_DATA = f"{ROOT}/data"
D_MSA  = f"{D_DATA}/msa"
D_RUNS = f"{ROOT}/runs/02_dtype_fix"
D_OUT  = f"{ROOT}/out"
D_LOGS = f"{ROOT}/logs"
for d in (D_DATA, D_MSA, D_RUNS, D_OUT, D_LOGS):
    os.makedirs(d, exist_ok=True)
L_W = "/content/esmfold2_weights"

if not os.path.exists(f"{D_W}/config.json"):
    raise SystemExit(
        f"DỪNG: không thấy {D_W}/config.json → cache weights KHÔNG có trên Drive.\n"
        f"Notebook 02 cố tình KHÔNG tải weights (24.9 GB, 230 s). Chạy lại cell 3 của "
        f"notebook 01 để dựng cache, rồi quay lại đây.")

drive_free_gb   = shutil.disk_usage(ROOT).free / 1024**3
content_free_gb = shutil.disk_usage("/content").free / 1024**3
ENV.update(drive_free_gb=round(drive_free_gb, 1), content_free_gb=round(content_free_gb, 1))
print(f"Drive  : còn {drive_free_gb:.1f} GB · /content: còn {content_free_gb:.1f} GB")
if drive_free_gb < 2.0:
    raise SystemExit(f"DỪNG: Drive chỉ còn {drive_free_gb:.1f} GB, không ghi nổi output. Dọn Drive.")
if STAGE_TO_LOCAL and content_free_gb < 30.0:
    print(f"CẢNH BÁO: /content còn {content_free_gb:.1f} GB → tắt STAGE_TO_LOCAL, load trực tiếp từ Drive.")
    STAGE_TO_LOCAL = False

try:
    n_cpu = os.cpu_count()
    with open("/proc/meminfo") as f:
        ram_gb = int([l for l in f if l.startswith("MemTotal")][0].split()[1]) / 1024**2
except Exception:
    n_cpu, ram_gb = None, None
ENV.update(cpu_count=n_cpu, ram_total_gb=None if ram_gb is None else round(ram_gb, 1))
print(f"CPU/RAM: {n_cpu} core · {ram_gb:.0f} GB RAM")
if ram_gb is not None and ram_gb < 40:
    print("CẢNH BÁO: < 40 GB RAM. dtype=None giữ checkpoint fp32 ở dạng mmap nên RSS thấp, "
          "nhưng set_esmc_precision('fp8') cần ~13 GB RAM thật. Nếu runtime bị kill lúc load → báo Claude.")

REPORT = {
    "notebook": "02_dtype_fix_and_measure",
    "schema": 1,
    "started_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "env": ENV,
    "root_cause_hypothesis": {
        "summary": ("forward() dong autocast bf16 tai model.py:1160 ngay sau `z = z.float()` (1159); "
                    "distogram_head (1181) / structure_head.sample (1185) / confidence_head (1239) "
                    "chay NGOAI autocast tren z fp32 => weight 3 head do PHAI la fp32. "
                    "from_pretrained(dtype=torch.bfloat16) cast ca 3 head sang bf16 "
                    "=> 'mat1 Float, mat2 BFloat16' tai distogram_head."),
        "implication_1": "diffusion decoder CHUA TUNG chay trong notebook 01 => moi so VRAM/time la san duoi",
        "implication_2": "esmc_precision khong lien quan; dtype=bf16 + esmc bf16 se chet y nguyen",
        "fix": "dtype=None (fp32). Trunk van bf16 nho autocast model.py:973 + FoldingTrunk cast noi bo layers.py:2839",
        "why_fp32_is_complete": ("InputsEmbedder tra ve torch.cat([a_bf16, aatype_fp32, profile_fp32, "
                                "deletion_mean_fp32]) (layers.py:2136, model.py:936/976/977) -> type "
                                "promotion lam x_inputs = fp32. x_inputs la s_inputs cua structure_head "
                                "(1187) va confidence_head (1240). Vay CA HAI dau vao cua 3 head (z va "
                                "x_inputs) deu fp32 co chu y => load fp32 khop hoan toan, khong con lech dtype."),
    },
    "knobs": {
        "model_dtype": str(MODEL_DTYPE), "esmc_precision": ESMC_PRECISION,
        "gate_num_loops": GATE_NUM_LOOPS, "gate_num_sampling_steps": GATE_SAMPLING_STEPS,
        "gate_lm_dropout": GATE_LM_DROPOUT,
        "probe_num_loops": PROBE_NUM_LOOPS, "probe_num_sampling_steps": PROBE_SAMPLING_STEPS,
        "mt_num_loops": MT_NUM_LOOPS, "mt_num_sampling_steps": MT_SAMPLING_STEPS,
        "msa_max_depth": MSA_MAX_DEPTH, "msa_column_mask_rate": MSA_COLUMN_MASK_RATE,
        "pae_cutoff": PAE_CUTOFF, "dist_cutoff": DIST_CUTOFF,
        "hard_cap_min": HARD_CAP_MIN, "alloc_conf": os.environ.get("PYTORCH_CUDA_ALLOC_CONF"),
        "stage_to_local": STAGE_TO_LOCAL,
    },
    "open_discrepancies_vs_config_json": {
        "num_sampling_steps": {"fold_default": 200, "config_json_inference_num_steps": 14,
                               "used_here": GATE_SAMPLING_STEPS, "must_ask_organizer": True},
        "lm_dropout": {"fold_default": 0.3, "config_json_lm_encoder_lm_dropout": 0.25,
                       "used_here": GATE_LM_DROPOUT, "stochastic": True,
                       "fold_docstring": "paper folding-eval value; 0/None disables",
                       "must_ask_organizer": True},
    },
    "steps": {}, "configs_tried": [], "folds": [], "ceiling_probe": [],
    "errors": [],
    "manual_fill": {"compute_units_remaining_before_run": "<<DIEN TAY>>",
                    "compute_units_remaining_after_run": "<<DIEN TAY>>"},
}
PARTIAL = f"{D_OUT}/02_partial.json"

def save_partial():
    REPORT["elapsed_min"] = round(elapsed_min(), 2)
    REPORT["cu_estimate"] = round(cu_burned(), 3)
    tmp = PARTIAL + ".tmp"
    with open(tmp, "w") as f:
        json.dump(REPORT, f, indent=2, default=str)
    os.replace(tmp, PARTIAL)

save_partial()
print(f"\nBáo cáo từng phần: {PARTIAL}  (ghi lại sau MỖI bước)")
print("TIỀN KIỂM: OK.")
budget("(cell 1 xong)")

## Cell 2 — cài `esm==3.4.3`

Notebook 01 đã xác nhận: cài sạch, **torch không bị thay** (2.14.1+cu130 giữ nguyên).
Cell này vẫn chặn cứng nếu pip lỡ đổi torch, vì CUDA lệch là mất cả buổi.

In [ ]:
# ============================== CELL 2: INSTALL esm ==============================
budget("(bắt đầu cell 2)")
t = time.time()
torch_before = torch.__version__

try:
    import esm as _probe
    have = getattr(_probe, "__version__", "unknown")
except Exception:
    have = None

if have == "3.4.3":
    print("esm 3.4.3 đã có sẵn — bỏ qua pip.")
    install_s = 0.0
else:
    print(f"esm hiện tại: {have} → cài esm==3.4.3 (wheel ~4.6 GB deps, 3–6 phút) ...", flush=True)
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "esm==3.4.3"],
                       capture_output=True, text=True)
    install_s = time.time() - t
    print(r.stdout[-3000:]); print(r.stderr[-3000:])
    if r.returncode != 0:
        REPORT["errors"].append({"step": "install", "rc": r.returncode, "stderr": r.stderr[-3000:]})
        save_partial()
        raise SystemExit("DỪNG: pip install esm==3.4.3 lỗi. Copy output đỏ ở trên gửi Claude.")

import importlib
importlib.invalidate_caches()
import torch as _t2
torch_after = _t2.__version__
if torch_after != torch_before:
    REPORT["errors"].append({"step": "install", "torch_changed": [torch_before, torch_after]})
    save_partial()
    raise SystemExit(
        f"DỪNG: pip đã đổi torch {torch_before} -> {torch_after}. CUDA có thể lệch.\n"
        f"Runtime > Restart session, rồi Run all lại từ cell 1.")

from esm.models.esmfold2 import (
    EsmFold2Model, ESMFold2InputBuilder, ProteinInput, StructurePredictionInput, MSA,
)
from esm.utils.msa.msa import is_a3m_insertion
import esm as _esm
print(f"\nesm {getattr(_esm, '__version__', '?')} · torch {torch_after} (không đổi) · "
      f"cài trong {install_s:.0f}s")
REPORT["steps"]["install"] = {"esm_version": getattr(_esm, "__version__", None),
                              "torch_before": torch_before, "torch_after": torch_after,
                              "seconds": round(install_s, 1)}
save_partial()
budget("(cell 2 xong)")

## Cell 3 — VERIFY cache weights (tải 0 byte)

Notebook 01 đã tải 24.901 GB trong 230.8 s và stage 127.9 s. **Không tải lại.**
Cell này chỉ kiểm từng byte; thiếu file → dừng cứng và chỉ đúng cách sửa.

In [ ]:
# ============================== CELL 3: VERIFY WEIGHTS CACHE (no download) =====
budget("(bắt đầu cell 3)")

MANIFEST = {
    "config.json":                      8214,
    "model.safetensors.index.json":     245594,
    "model-00001-of-00006.safetensors": 4955672788,
    "model-00002-of-00006.safetensors": 4971194264,
    "model-00003-of-00006.safetensors": 4997460576,
    "model-00004-of-00006.safetensors": 4979069192,
    "model-00005-of-00006.safetensors": 4934556472,
    "model-00006-of-00006.safetensors": 1481396616,
    "ccd.pkl":                          417306584,
}
MANIFEST_TOTAL = sum(MANIFEST.values())

def audit(d):
    ok, bad = [], []
    for fn, size in MANIFEST.items():
        p = os.path.join(d, fn)
        if not os.path.exists(p):
            bad.append((fn, "missing", 0, size))
        else:
            got = os.path.getsize(p)
            (ok if got == size else bad).append((fn, "ok" if got == size else "size-mismatch", got, size))
    return ok, bad

def header_sane(p):
    try:
        with open(p, "rb") as f:
            n = int.from_bytes(f.read(8), "little")
        return 0 < n < 50_000_000 and n + 8 <= os.path.getsize(p)
    except Exception:
        return False

ok, bad = audit(D_W)
print(f"Cache Drive {D_W}: {len(ok)}/{len(MANIFEST)} file đúng byte "
      f"({MANIFEST_TOTAL/1024**3:.2f} GB).")
if bad:
    for fn, why, got, want in bad:
        print(f"  THIẾU/SAI  {fn:38s} {why:14s} got={got} want={want}")
    REPORT["errors"].append({"step": "verify_cache", "bad": bad})
    save_partial()
    raise SystemExit(
        "DỪNG: cache weights KHÔNG đầy đủ. Notebook 02 cố tình không tải (24.9 GB).\n"
        "Cách sửa: mở notebook 01, chạy cell 1 + cell 3 (cell 3 sẽ tải đúng các file thiếu "
        "ở trên, không tải lại file đã đúng), rồi quay lại notebook 02.")

with open(os.path.join(D_W, "model.safetensors.index.json")) as f:
    shards = sorted(set(json.load(f)["weight_map"].values()))
missing = [s for s in shards if s not in MANIFEST]
if missing:
    REPORT["errors"].append({"step": "verify_cache", "index_refs_missing": missing})
    save_partial()
    raise SystemExit(f"DỪNG: index.json trỏ tới shard không có: {missing}")
print(f"index.json: {len(shards)} shard, tất cả đều có.")

MODEL_DIR = D_W
stage_s = 0.0
if STAGE_TO_LOCAL:
    os.makedirs(L_W, exist_ok=True)
    _, bad_l = audit(L_W)
    if bad_l:
        print(f"\nCopy Drive → {L_W} ({MANIFEST_TOTAL/1024**3:.1f} GB, ~2–4 phút; "
              f"mmap qua Drive FUSE không đáng tin khi đọc 26 GB) ...")
        t = time.time()
        for fn, why, got, want in bad_l:
            print(f"  {fn} ...", flush=True)
            shutil.copy2(os.path.join(D_W, fn), os.path.join(L_W, fn))
        stage_s = time.time() - t
        _, bad_l = audit(L_W)
        if bad_l:
            REPORT["errors"].append({"step": "stage_local", "bad": bad_l})
            save_partial()
            raise SystemExit(f"DỪNG: copy sang /content lỗi: {bad_l}")
        print(f"  xong {stage_s:.0f}s")
    else:
        print(f"\n{L_W} đã đủ weights — bỏ qua copy.")
    MODEL_DIR = L_W

bad_hdr = [fn for fn in MANIFEST if fn.endswith(".safetensors")
           and not header_sane(os.path.join(MODEL_DIR, fn))]
if bad_hdr:
    REPORT["errors"].append({"step": "verify_cache", "bad_header": bad_hdr})
    save_partial()
    raise SystemExit(f"DỪNG: header safetensors sai: {bad_hdr}")

CCD_DIR = MODEL_DIR
print(f"\nMODEL_DIR = {MODEL_DIR}   (tải 0 byte)")
REPORT["steps"]["weights"] = {
    "model_dir_used": MODEL_DIR, "downloaded_bytes": 0,
    "files_verified": len(MANIFEST), "manifest_total_gb": round(MANIFEST_TOTAL / 1024**3, 3),
    "stage_to_local_seconds": round(stage_s, 1),
}
save_partial()
budget("(cell 3 xong)")

## Cell 4 — sequence

- **IL-7Rα** `P16871` Glu21–Asp239 = 219 aa (đúng range cuộc thi)
- **TL1A** `O95150` Leu72–Leu251 = 180 aa / protomer
- **Binder (+)**: minibinder de novo 55 aa, PDB **7S5B / 7OPB** chain D-E-F, Cao et al.
  *Nature* 2022 — **đã validate thực nghiệm, ái lực picomolar**. Đây là mẫu DƯƠNG.
- **Binder (−)**: xáo trộn chính sequence trên, `random.Random(20261111)` → tái lập được.
  Cùng thành phần amino acid, không còn cấu trúc. Đây là mẫu ÂM.
- **DcR3** `O95407` res30–205 ecto = 176 aa — **receptor tự nhiên của TL1A**, có tinh thể
  phức (PDB **3K51**). Dùng cho câu hỏi monomer-vs-trimer, vì receptor họ TNF bám vào
  **rãnh giữa hai protomer kề nhau** → nếu monomer và trimer cho ipSAE khác nhau, nó sẽ
  khác nhau ở đây. **Không phải** candidate để submit.

In [ ]:
# ============================== CELL 4: SEQUENCES ==============================
budget("(bắt đầu cell 4)")
import random

IL7RA = (
    "ESGYAQNGDLEDAELDDYSFSCYSQLEVNGSQHSLTCAFEDPDVNITNLEFEICGALVEVKCLNFRKLQEIYFIETK"
    "KFLLIGKSNICVKVGEKSLTCKKIDLTTIVKPEAPFDLSVVYREGANDFVVTFNTSHLQKKYVKVLMHDVAYRQEKD"
    "ENKWTHVNLSSTKLTLLQRKLQPAAMYEIKVRSIPDHYFKGFWSEWSPSYYFRTPEINNSSGEMD"
)
TL1A = (
    "LKGQEFAPSHQQVYAPLRADGDKPRAHLTVVRQTPTQHFKNQFPALHWEHELGLAFTKNRMNYTNKFLLIPESGDYF"
    "IYSQVTFRGMTSECSEIRQAGRPNKPDSITVVITKVTDSYPEPTQLLMGTKSVCEVGSNWFQPIYLGAMFSLQEGDK"
    "LMVNVSDISLVDYTKEDKTFFGAFLL"
)
BINDER_POS = "SVIEKLRKLEKQARKQGDEVLVMLARMVLEYLEKGWVSEEDADESADRIEEVLKK"   # 7S5B/7OPB D-F
DCR3 = (
    "VAETPTYPWRDAETGERLVCAQCPPGTFVQRPCRRDSPTTCGPCPPRHYTQFWNYLERCRYCNVLCGEREEEARACH"
    "ATHNRACRCRTGFFAHAGFCLEHASCPPGAGVIAPGTPSQNTQCQPCPPGTFSASSSSSEQCQPHRNCTALGLALNV"
    "PGSSSHDTLCTSCTGFPLSTRV"
)

assert len(IL7RA) == 219 and len(TL1A) == 180 and len(BINDER_POS) == 55 and len(DCR3) == 176
assert IL7RA.startswith("ESG") and IL7RA.endswith("GEMD")
assert TL1A.startswith("LKGQ") and TL1A.endswith("AFLL")
assert DCR3.startswith("VAET") and DCR3.endswith("STRV")

_rng = random.Random(20261111)
_c = list(BINDER_POS); _rng.shuffle(_c)
BINDER_NEG = "".join(_c)
assert sorted(BINDER_NEG) == sorted(BINDER_POS) and BINDER_NEG != BINDER_POS

BINDER_60  = (BINDER_POS * 2)[:60]
BINDER_100 = (BINDER_POS * 2)[:100]
BINDER_235 = (BINDER_POS * 5)[:235]

# fold tối thiểu để chốt dtype: nhỏ nhất có thể mà vẫn là phức 2 chain thật
MIN_A, MIN_B = IL7RA[:60], BINDER_POS[:20]

print(f"IL-7Ra      {len(IL7RA):4d} aa")
print(f"TL1A        {len(TL1A):4d} aa / protomer")
print(f"binder (+)  {len(BINDER_POS):4d} aa  {BINDER_POS}      <- 7S5B, validate thực nghiệm")
print(f"binder (-)  {len(BINDER_NEG):4d} aa  {BINDER_NEG}      <- xáo trộn, seed 20261111")
print(f"DcR3 ecto   {len(DCR3):4d} aa   <- receptor TL1A tự nhiên (PDB 3K51), chỉ dùng làm PROBE")
print(f"\nPhức cửa sàng : IL-7Ra + binder = {len(IL7RA)+len(BINDER_POS)} residue")
print(f"Fold tối thiểu: {len(MIN_A)} + {len(MIN_B)} = {len(MIN_A)+len(MIN_B)} residue")
print(f"DcR3 + TL1A x1 = {len(TL1A)+len(DCR3)} · DcR3 + TL1A x3 = {3*len(TL1A)+len(DCR3)} residue")

REPORT["steps"]["sequences"] = {
    "il7ra_len": len(IL7RA), "tl1a_len": len(TL1A), "dcr3_len": len(DCR3),
    "binder_pos": BINDER_POS,
    "binder_pos_source": "PDB 7S5B / 7OPB chain D-F, Cao et al. Nature 2022, picomolar, VALIDATED",
    "binder_neg": BINDER_NEG, "binder_neg_rule": "random.Random(20261111).shuffle(binder_pos)",
    "dcr3_source": "UniProt O95407 res30-205 ectodomain approx; complex PDB 3K51; PROBE not candidate",
    "gate_complex_residues": len(IL7RA) + len(BINDER_POS),
}
save_partial()
budget("(cell 4 xong)")

## Cell 5 — MSA THẬT, nhúng sẵn trong notebook (không phụ thuộc mạng)

Notebook 01 rơi về `query_only_FALLBACK` (depth 1) và `fetch_info` rỗng → không tái dựng
được cửa sàng. Đã điều tra: **ColabFold MSA API vẫn chạy bình thường.** Gọi lại cùng
protocol (`POST /ticket/msa` với `q=">101\n<seq>\n"`, `mode=env` → poll `/ticket/<id>` →
`GET /result/download/<id>` → `uniref.a3m`) trả về:

| target | depth | match columns | thời gian |
|---|---|---|---|
| IL-7Rα 219 aa | **921** | 219 ✓ | 7 s |
| TL1A 180 aa | **2037** | 180 ✓ | 29 s |

Cả hai a3m đã **nhúng gzip+base64 vào cell này**, kèm `sha256` để verify. Nghĩa là
notebook không còn cần mạng cho đường quan trọng nhất; nếu nhúng lỗi thì mới gọi API,
và lúc đó ghi lại **nguyên request/response** thay vì thất bại âm thầm như lần trước.

Hàng query của cả hai a3m **khớp đúng từng ký tự** với sequence target → `prepare_input`
sẽ không lệch index.

> Lưu ý: `mode=env` trả 3 file; ta dùng `uniref.a3m`. Depth 921/2037 đã quá
> `msa_max_depth=1024` nên `fold()` còn subsample lại mỗi loop.

In [ ]:
# ============================== CELL 5: MSA (embedded, real) ==================
budget("(bắt đầu cell 5)")
import gzip, base64, hashlib, urllib.request, urllib.parse, tarfile, io as _io

A3M_META = {
    "IL7RA": {"name": "IL7Ra_P16871_E21-D239", "seq_len": 219, "depth": 921,
              "sha256": "08e2a05b5b4787bb28c03302b1d56c3856980423a5582f2096bc6f5238b0be03"},
    "TL1A":  {"name": "TL1A_O95150_L72-L251",  "seq_len": 180, "depth": 2037,
              "sha256": "4464ad15470e2ecda72ea879beb5972059062085b04ff515cef26f642e092c27"},
}
A3M_SEQ = {"IL7RA": IL7RA, "TL1A": TL1A}

# ---- payload nhúng: gzip + base64 của uniref.a3m lấy từ api.colabfold.com (mode=env) ----
A3M_B64 = {}
A3M_B64["IL7RA"] = (
    "..."
)

A3M_B64["TL1A"] = (
    "..."
)

HOST = "https://api.colabfold.com"

def _req(url, data=None, timeout=120):
    body = urllib.parse.urlencode(data).encode() if data else None
    return urllib.request.urlopen(
        urllib.request.Request(url, data=body, headers={"User-Agent": UA}), timeout=timeout)

def fetch_colabfold_a3m(seq, out_path, deadline_s=600):
    """Trả (ok, info). info LUÔN có nội dung — lần trước nó rỗng nên không debug được."""
    info = {"host": HOST, "mode": "env", "seq_len": len(seq), "stages": []}
    try:
        t0 = time.time()
        raw = _req(f"{HOST}/ticket/msa", {"q": f">101\n{seq}\n", "mode": "env"}, 60).read()
        info["stages"].append({"stage": "submit", "response": raw[:300].decode("utf8", "replace")})
        tk = json.loads(raw); tid, st = tk.get("id"), tk.get("status")
        info["ticket_id"], info["submit_status"] = tid, st
        if not tid:
            info["fail"] = "khong co ticket id"; return False, info
        print(f"  ticket {tid} ({st}) ...", flush=True)
        while st in ("UNKNOWN", "RUNNING", "PENDING", "RATELIMIT"):
            if time.time() - t0 > deadline_s:
                info["fail"] = f"timeout {deadline_s}s o status {st}"; return False, info
            time.sleep(8)
            st = json.loads(_req(f"{HOST}/ticket/{tid}").read()).get("status")
            print(f"    {st}  ({time.time()-t0:.0f}s)", flush=True)
        info["final_status"] = st
        if st != "COMPLETE":
            info["fail"] = f"status cuoi = {st}"; return False, info
        blob = _req(f"{HOST}/result/download/{tid}", timeout=600).read()
        info["tar_bytes"] = len(blob)
        with tarfile.open(fileobj=_io.BytesIO(blob)) as tar:
            info["members"] = tar.getnames()
            m = next((n for n in tar.getnames() if n.endswith("uniref.a3m")), None)
            if m is None:
                info["fail"] = "tar khong co uniref.a3m"; return False, info
            data = tar.extractfile(m).read()
        with open(out_path, "wb") as f:
            f.write(data)
        info["seconds"] = round(time.time() - t0, 1)
        info["a3m_bytes"] = len(data)
        return True, info
    except Exception as e:
        info["fail"] = f"{type(e).__name__}: {e}"
        return False, info

def a3m_stats(path, seq):
    raw = open(path, "rb").read()
    lines = [l for l in raw.decode("utf8", "replace").split("\n") if l and not l.startswith("#")]
    hdr = [i for i, l in enumerate(lines) if l.startswith(">")]
    q = lines[hdr[0] + 1] if hdr else ""
    match = sum(1 for c in q if not is_a3m_insertion(c))
    plain = "".join(c for c in q if not is_a3m_insertion(c)).replace("-", "")
    return {"bytes": len(raw), "sha256": hashlib.sha256(raw).hexdigest(),
            "depth": len(hdr), "match_columns": match,
            "query_matches_target": plain == seq, "first_header": lines[hdr[0]][:40] if hdr else None}

MSA_OBJ, MSA_REPORT = {}, {}
for key, meta in A3M_META.items():
    seq = A3M_SEQ[key]
    path = f"{D_MSA}/{meta['name']}.uniref.a3m"
    src, fetch_info = None, {}

    # 1) cache Drive (verify sha256; sai sha -> coi như không có)
    if os.path.exists(path):
        st = a3m_stats(path, seq)
        if st["sha256"] == meta["sha256"]:
            src = "drive_cache_sha_verified"
        elif st["match_columns"] == meta["seq_len"] and st["query_matches_target"] and st["depth"] >= 50:
            src = "drive_cache_other_but_valid"
        else:
            print(f"  {key}: a3m trên Drive KHÔNG hợp lệ (sha/{st['sha256'][:12]} "
                  f"depth={st['depth']} match={st['match_columns']}) -> bỏ, dựng lại")
            os.remove(path)

    # 2) payload nhúng trong notebook
    if src is None and key in A3M_B64:
        raw = gzip.decompress(base64.b64decode(A3M_B64[key]))
        got = hashlib.sha256(raw).hexdigest()
        if got == meta["sha256"]:
            with open(path, "wb") as f:
                f.write(raw)
            src = "embedded_in_notebook"
        else:
            print(f"  {key}: payload nhúng sai sha256 ({got[:12]} != {meta['sha256'][:12]}) -> gọi API")

    # 3) ColabFold API
    if src is None:
        print(f"  {key}: gọi ColabFold MSA API ...")
        ok, fetch_info = fetch_colabfold_a3m(seq, path, 600)
        src = "colabfold_api" if ok else None
        if not ok:
            print(f"    THẤT BẠI: {fetch_info}")

    # 4) hết đường -> a3m chỉ có query, GHI RÕ là không thật
    if src is None:
        with open(path, "w") as f:
            f.write(f">{meta['name']}_query_only\n{seq}\n")
        src = "query_only_FALLBACK"

    st = a3m_stats(path, seq)
    m = MSA.from_a3m(path=path)
    realistic = (m.depth >= 50)
    MSA_OBJ[key] = m
    MSA_REPORT[key] = {"source": src, "path": path, "msa_depth_loaded": m.depth,
                       "msa_depth_realistic": bool(realistic),
                       "sha256_matches_expected": st["sha256"] == meta["sha256"],
                       "fetch_info": fetch_info, **st}
    flag = "OK" if realistic else "KHONG THAT"
    print(f"{key:6s} src={src:28s} depth={m.depth:5d}  match_cols={st['match_columns']:4d}"
          f" (cần {meta['seq_len']:3d})  query==target={st['query_matches_target']}  [{flag}]")

    if st["match_columns"] != meta["seq_len"]:
        REPORT["errors"].append({"step": "msa", "key": key,
                                 "msg": f"match_columns {st['match_columns']} != {meta['seq_len']}"})
        REPORT["steps"]["msa"] = MSA_REPORT; save_partial()
        raise SystemExit(f"DỪNG: a3m {key} có {st['match_columns']} match column nhưng target "
                         f"{meta['seq_len']} aa → prepare_input sẽ lỗi index. Xoá {path} rồi chạy lại cell 5.")

MSA_IL7RA, MSA_TL1A = MSA_OBJ["IL7RA"], MSA_OBJ["TL1A"]
MSA_ALL_REALISTIC = all(v["msa_depth_realistic"] for v in MSA_REPORT.values())
print(f"\nmsa_depth_realistic (cả hai target) = {MSA_ALL_REALISTIC}")
if not MSA_ALL_REALISTIC:
    print("!!! MSA không thật → mọi ipSAE dưới đây là SÀN DƯỚI, không phải cửa sàng. Báo Claude.")

REPORT["steps"]["msa"] = MSA_REPORT
REPORT["steps"]["msa_all_realistic"] = bool(MSA_ALL_REALISTIC)
save_partial()
budget("(cell 5 xong)")

## Cell 6 — load model với dtype ĐÚNG + micro-repro chứng minh nguyên nhân

`dtype=None` → `_load_pretrained` **bỏ qua** `model.to(dtype)` (`hub.py:194`), giữ fp32.
Rồi `set_esmc_precision("fp8")` vẫn cast ESMC sang bf16→fp8, nên ESMC **không** tốn 25 GB.

Trunk vẫn chạy bf16: autocast ở `model.py:973` bọc toàn bộ trunk, và
`FoldingTrunk.forward` tự cast pair sang bf16 khi bật fused kernel (`layers.py:2839`),
weight thì được cast tại call site (`_to_fused_kernel_dtype`, `layers.py:2625`).

Sau khi load, cell in dtype của từng khối và chạy **micro-repro**: lấy chính
`distogram_head`, clone sang bf16, bắn vào nó một tensor fp32 → tái tạo **đúng** lỗi của
notebook 01 trong vài milli-giây. Đó là bằng chứng, không phải suy diễn.

In [ ]:
# ============================== CELL 6: LOAD (fp32) + ROOT-CAUSE REPRO ========
budget("(bắt đầu cell 6)")
import gc, copy, warnings, traceback

def vram():
    return (torch.cuda.memory_allocated() / 1024**3,
            torch.cuda.max_memory_allocated() / 1024**3,
            torch.cuda.max_memory_reserved() / 1024**3)

def rss_gb():
    try:
        with open("/proc/self/status") as f:
            for l in f:
                if l.startswith("VmRSS"):
                    return round(int(l.split()[1]) / 1024**2, 2)
    except Exception:
        return None

model = None
def load_model(dtype, esmc_precision, kernel_backend="fused"):
    """Load model, trả (model, info). Giải phóng model cũ trước."""
    global model
    if model is not None:
        del model; model = None
        gc.collect(); torch.cuda.empty_cache()
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    dt = None if dtype is None else getattr(torch, dtype)
    print(f"Load dtype={dtype} esmc_precision={esmc_precision} ... (~2 phút)", flush=True)
    t = time.time()
    m = EsmFold2Model.from_pretrained(
        MODEL_DIR, device="cuda", dtype=dt, esmc_precision=esmc_precision, load_esmc=True,
    ).eval()
    load_s = time.time() - t
    kb = kernel_backend
    try:
        m.set_kernel_backend(kernel_backend)
    except Exception as e:
        kb = None; m.set_kernel_backend(None)
        print(f"CẢNH BÁO: không bật được kernel '{kernel_backend}' ({type(e).__name__}: {e}) -> reference path")
    alloc, peak, reserved = vram()
    info = {
        "dtype": str(dtype), "esmc_precision": esmc_precision,
        "kernel_backend_effective": kb,
        "seconds": round(load_s, 1),
        "vram_weights_gb": round(alloc, 3), "vram_peak_during_load_gb": round(peak, 3),
        "vram_reserved_gb": round(reserved, 3),
        "vram_free_after_load_gb": round(torch.cuda.mem_get_info(0)[0] / 1024**3, 3),
        "ram_rss_gb": rss_gb(),
        "n_params": sum(p.numel() for p in m.parameters()),
        "n_buffer_elements": sum(b.numel() for b in m.buffers()),
        "msa_encoder_present": m.msa_encoder is not None,
        "config_num_loops": m.config.num_loops,
        "config_inference_num_steps": getattr(m.config.structure_head, "inference_num_steps", None),
        "config_lm_dropout": getattr(m.config.lm_encoder, "lm_dropout", None),
    }
    model = m
    return m, info

MODEL_DTYPE_STR = None if MODEL_DTYPE is None else str(MODEL_DTYPE)
model, LOAD_INFO = load_model(MODEL_DTYPE_STR, ESMC_PRECISION)
KERNEL_BACKEND = LOAD_INFO["kernel_backend_effective"]

# ---------- dtype của từng khối: đây là bản đồ của cả vấn đề ----------
def dt_of(mod, attr_path):
    o = mod
    for a in attr_path.split("."):
        o = getattr(o, a, None)
        if o is None: return None
    try:
        return str(next(o.parameters()).dtype)
    except Exception:
        return str(getattr(o, "dtype", None))

DTYPE_MAP = {
    "folding_trunk (TRUNK, chay trong autocast)":        dt_of(model, "folding_trunk"),
    "lm_encoder":                                        dt_of(model, "lm_encoder"),
    "msa_encoder":                                       dt_of(model, "msa_encoder"),
    "parcae_coda":                                       dt_of(model, "parcae_coda"),
    "distogram_head  (NGOAI autocast, can fp32)":        str(model.distogram_head.weight.dtype),
    "structure_head  (NGOAI autocast, can fp32)":        dt_of(model, "structure_head"),
    "confidence_head (NGOAI autocast, can fp32)":        dt_of(model, "confidence_head"),
    "esmc (fp8 -> weight nam trong buffer)":             dt_of(model, "esmc"),
}
print("\n--- dtype thực tế của từng khối ---")
for k, v in DTYPE_MAP.items():
    print(f"  {k:48s} {v}")

print(f"\nLoad {LOAD_INFO['seconds']:.0f}s · VRAM weights {LOAD_INFO['vram_weights_gb']:.3f} GB "
      f"· còn trống {LOAD_INFO['vram_free_after_load_gb']:.2f} GB · RSS {LOAD_INFO['ram_rss_gb']} GB")
print(f"params {LOAD_INFO['n_params']:,} + {LOAD_INFO['n_buffer_elements']:,} buffer element")
print(f"kernel backend = {KERNEL_BACKEND!r} · msa_encoder = {LOAD_INFO['msa_encoder_present']}")
print(f"config.json: num_loops={LOAD_INFO['config_num_loops']} "
      f"inference_num_steps={LOAD_INFO['config_inference_num_steps']} "
      f"lm_encoder.lm_dropout={LOAD_INFO['config_lm_dropout']}")
if not LOAD_INFO["msa_encoder_present"]:
    print("\n!!! checkpoint KHÔNG có MSA encoder → mọi MSA bị bỏ qua âm thầm. Báo Claude ngay.")

# ---------- MICRO-REPRO: 0 giây GPU, chứng minh nguyên nhân gốc ----------
print("\n--- micro-repro: dòng 1181 model.py, distogram_head(z_fp32) ---")
lin = model.distogram_head
x32 = torch.zeros(1, 4, 4, lin.in_features, device="cuda", dtype=torch.float32)
repro = {"distogram_head_weight_dtype_as_loaded": str(lin.weight.dtype)}

try:
    _ = lin(x32); repro["fp32_head_on_fp32_input"] = "OK"
except Exception as e:
    repro["fp32_head_on_fp32_input"] = f"{type(e).__name__}: {e}"

lin_bf16 = copy.deepcopy(lin).to(torch.bfloat16)
try:
    _ = lin_bf16(x32); repro["bf16_head_on_fp32_input"] = "OK (khong tai hien duoc!)"
except Exception as e:
    repro["bf16_head_on_fp32_input"] = f"{type(e).__name__}: {e}"
del lin_bf16, x32

repro["reproduces_notebook01_error"] = (
    "mat1 and mat2 must have the same dtype" in str(repro["bf16_head_on_fp32_input"]))
for k, v in repro.items():
    print(f"  {k:42s} {v}")
if repro["reproduces_notebook01_error"]:
    print("\n  => TÁI HIỆN ĐÚNG lỗi của notebook 01. Nguyên nhân gốc: dtype=bfloat16 cast "
          "distogram_head/structure_head/confidence_head sang bf16, nhưng forward() chạy chúng "
          "NGOÀI autocast trên z fp32 (model.py:1159-1241).")
else:
    print("\n  => KHÔNG tái hiện được. Nguyên nhân có thể khác — ghi lại và báo Claude.")

t = time.time()
builder = ESMFold2InputBuilder(ccd_cache=CCD_DIR)   # đọc <CCD_DIR>/ccd.pkl, không gọi mạng
ccd_s = time.time() - t
print(f"\nESMFold2InputBuilder OK (ccd.pkl đọc từ cache trong {ccd_s:.0f}s)")

LOAD_INFO["ccd_load_seconds"] = round(ccd_s, 1)
LOAD_INFO["dtype_map"] = DTYPE_MAP
REPORT["steps"]["load_model"] = LOAD_INFO
REPORT["steps"]["root_cause_repro"] = repro
REPORT["configs_tried"].append({"config": f"dtype={MODEL_DTYPE_STR}, esmc={ESMC_PRECISION}",
                                "load_ok": True, **{k: LOAD_INFO[k] for k in
                                ("seconds", "vram_weights_gb", "kernel_backend_effective")}})
save_partial()
budget("(cell 6 xong — model trên GPU)")

## Cell 7 — adapter ESMFold2 → ipSAE

Giữ nguyên adapter đã test offline ở notebook 01 (đường **AF2**: `.pdb` + `.json` có
`pae`/`plddt`/`iptm`/`ptm`; `token_chain_ids`/`token_res_ids` của đường AF3 **không được
ipsae.py đọc tới**). Ta tự ghi PDB **theo thứ tự token** nên permutation là identity
*by construction*; notebook 01 đã xác nhận `is_permutation=true, is_identity=true` cả 5 lần.

Ba assert chặn bẫy shape âm thầm: `n_CA == n_CB == n_residues == pae.shape[0]`.
pLDDT của ESMFold2 thang 0–1 → nhân 100 cho ipsae/pDockQ.

In [ ]:
# ============================== CELL 7: ipSAE ADAPTER ==========================
budget("(bắt đầu cell 7)")
import hashlib, numpy as np, urllib.request

IPSAE_PY = f"{D_DATA}/ipsae.py"
IPSAE_URL = "https://raw.githubusercontent.com/DunbrackLab/IPSAE/main/ipsae.py"
if not os.path.exists(IPSAE_PY) or os.path.getsize(IPSAE_PY) < 5000:
    print(f"Tải {IPSAE_URL}")
    with urllib.request.urlopen(
            urllib.request.Request(IPSAE_URL, headers={"User-Agent": UA}), timeout=60) as r, \
         open(IPSAE_PY, "wb") as f:
        f.write(r.read())
IPSAE_SHA = hashlib.sha256(open(IPSAE_PY, "rb").read()).hexdigest()
print(f"ipsae.py {IPSAE_PY}\n  sha256 = {IPSAE_SHA}")

def token_order_audit(spi):
    """perm[k] = token_index của residue thứ k theo thứ tự ghi ra PDB."""
    _f, cis = builder.prepare_input(spi, seed=0)      # device=None -> CPU, rẻ
    perm = []
    for ci in cis:
        seen = set()
        for tk in ci.tokens:
            if tk.residue_index in seen:
                continue
            seen.add(tk.residue_index)
            perm.append(tk.token_index)
    perm = np.asarray(perm, dtype=np.int64)
    n_tok = sum(len(ci.tokens) for ci in cis)
    info = {"n_tokens": int(n_tok), "n_residues_out": int(perm.size),
            "is_permutation": bool(np.array_equal(np.sort(perm), np.arange(perm.size))),
            "is_identity": bool(perm.size == n_tok and np.array_equal(perm, np.arange(perm.size))),
            "chain_order": [ci.chain_id for ci in cis],
            "chain_lengths": [len(ci.tokens) for ci in cis]}
    del _f, cis
    return perm, info

def write_pdb_token_order(mc, path):
    chain_lookup = mc.metadata.chain_lookup
    lines, serial, n_ca, n_cb, res_counter = [], 1, 0, 0, {}
    for tok_idx, (s, e) in enumerate(mc.token_to_atoms):
        resname = str(mc.sequence[tok_idx])[:3]
        chain = str(chain_lookup[int(mc.chain_id[tok_idx])])[:1] or "A"
        res_counter[chain] = res_counter.get(chain, 0) + 1
        resnum = res_counter[chain]
        b = float(mc.plddt[tok_idx]) * 100.0
        has_ca = has_cb = False
        for a in range(int(s), int(e)):
            an = str(mc.atom_names[a]).strip().upper()
            el = str(mc.atom_elements[a]).strip().upper()[:2]
            x, y, z = (float(v) for v in mc.atom_positions[a])
            an4 = f" {an:<3}" if len(an) < 4 else an[:4]
            lines.append(f"ATOM  {serial:5d} {an4} {resname:>3s} {chain:1s}{resnum:4d}    "
                         f"{x:8.3f}{y:8.3f}{z:8.3f}{1.00:6.2f}{b:6.2f}          {el:>2s}")
            serial += 1
            if an == "CA":
                has_ca = True
                if resname == "GLY":
                    has_cb = True
            if an == "CB":
                has_cb = True
        n_ca += has_ca; n_cb += has_cb
    with open(path, "w") as f:
        f.write("\n".join(lines + ["TER", "END"]) + "\n")
    return len(mc.token_to_atoms), n_ca, n_cb

def run_ipsae(result, tag, outdir, perm=None):
    os.makedirs(outdir, exist_ok=True)
    pdb_p, json_p = f"{outdir}/{tag}.pdb", f"{outdir}/{tag}.json"
    n_res, n_ca, n_cb = write_pdb_token_order(result.complex, pdb_p)

    pae = result.pae.detach().cpu().float().numpy()
    plddt = result.plddt.detach().cpu().float().numpy().ravel()
    if perm is not None and not np.array_equal(perm, np.arange(perm.size)):
        pae = pae[np.ix_(perm, perm)]; plddt = plddt[perm]
    if plddt.max() <= 1.0:
        plddt = plddt * 100.0

    assert pae.shape == (n_res, n_res), f"PAE {pae.shape} != ({n_res},{n_res})"
    assert n_ca == n_res, f"PDB có {n_ca} CA nhưng {n_res} residue"
    assert n_cb == n_res, f"PDB có {n_cb} CB(+GLY CA) nhưng {n_res} residue -> ipsae lệch shape"

    with open(json_p, "w") as f:
        json.dump({"pae": np.round(pae, 2).tolist(), "plddt": np.round(plddt, 2).tolist(),
                   "iptm": float(result.iptm) if result.iptm is not None else -1.0,
                   "ptm": float(result.ptm) if result.ptm is not None else -1.0}, f)

    r = subprocess.run([sys.executable, IPSAE_PY, json_p, pdb_p, str(PAE_CUTOFF), str(DIST_CUTOFF)],
                       capture_output=True, text=True, timeout=1800)
    txt_p = f"{outdir}/{tag}_{PAE_CUTOFF:02d}_{DIST_CUTOFF:02d}.txt"
    out = {"ipsae_returncode": r.returncode, "ipsae_stderr": r.stderr[-2000:],
           "pdb": pdb_p, "json": json_p, "txt": txt_p, "asym": {}, "max": {}}
    if r.returncode != 0 or not os.path.exists(txt_p):
        out["ipsae_stdout"] = r.stdout[-1500:]
        return out
    for line in open(txt_p):
        p = line.split()
        if len(p) < 6 or p[0] == "Chn1":
            continue
        try:
            val = float(p[5])
        except ValueError:
            continue
        if p[4] == "asym":
            out["asym"][f"{p[0]}->{p[1]}"] = val
        elif p[4] == "max":
            out["max"][f"{p[0]}-{p[1]}"] = val
    if out["asym"]:
        out["ipsae_min"] = min(out["asym"].values())
        out["ipsae_max"] = max(out["asym"].values())
    return out

print(f"\nAdapter sẵn sàng · cutoff {PAE_CUTOFF}/{DIST_CUTOFF} · cột 'ipSAE' (d0res) · đường AF2")
REPORT["steps"]["ipsae_adapter"] = {
    "ipsae_py": IPSAE_PY, "ipsae_sha256": IPSAE_SHA, "ipsae_url": IPSAE_URL,
    "pae_cutoff": PAE_CUTOFF, "dist_cutoff": DIST_CUTOFF,
    "path": "AF2 (.pdb + .json with pae/plddt/iptm/ptm)",
    "pdb_order": "model token order -> permutation la identity by construction",
}
save_partial()
budget("(cell 7 xong)")

## Cell 8 — hàm `fold_once`: đo VRAM + thời gian THẬT, fail-soft

Khác notebook 01 ở ba chỗ:

1. **Lưu `result.pae` ra `.npy`** cho mỗi fold → cell tính-tiền-định so sánh được từng bit.
2. **Bắt OOM riêng** khỏi lỗi khác, và dọn sạch allocator sau mỗi lần OOM.
3. `activation_gb = peak − weights` giờ **có cả diffusion decoder** (notebook 01 không có).

In [ ]:
# ============================== CELL 8: fold_once ==============================
budget("(bắt đầu cell 8)")

def build_spi(chains):
    """chains = [(chain_id | [ids], sequence, msa_or_None)]"""
    return StructurePredictionInput(sequences=[
        ProteinInput(id=cid, sequence=seq, msa=msa) for cid, seq, msa in chains])

def fold_once(tag, chains, **kw):
    """Trả dict kết quả. KHÔNG raise. LUÔN ghi vào REPORT['folds'] + save_partial()."""
    rec = _fold_once_inner(tag, chains, **kw)
    REPORT["folds"].append(rec)
    save_partial()
    return rec

def _fold_once_inner(tag, chains, num_loops, num_sampling_steps, seed=0,
                     lm_dropout=None, offload_esmc=False, chunk_size="default",
                     do_ipsae=True, save_pae=True, outdir=None):
    outdir = outdir or f"{D_RUNS}/{tag}"
    os.makedirs(outdir, exist_ok=True)
    lm_dropout = GATE_LM_DROPOUT if lm_dropout is None else lm_dropout
    n_res = sum((len(cid) if isinstance(cid, list) else 1) * len(seq) for cid, seq, _ in chains)
    rec = {"tag": tag, "n_residues": n_res,
           "chains": [{"id": cid, "len": len(seq),
                       "copies": len(cid) if isinstance(cid, list) else 1,
                       "msa_depth": (None if msa is None else msa.depth)} for cid, seq, msa in chains],
           "num_loops": num_loops, "num_sampling_steps": num_sampling_steps,
           "num_diffusion_samples": 1, "seed": seed, "lm_dropout": lm_dropout,
           "offload_esmc": bool(offload_esmc), "chunk_size": str(chunk_size),
           "msa_used": any(msa is not None for _, _, msa in chains),
           "kernel_backend": KERNEL_BACKEND, "ok": False}

    print(f"\n{'='*72}\n[{tag}] {n_res} residue · loops={num_loops} steps={num_sampling_steps} "
          f"· MSA={rec['msa_used']} · offload={offload_esmc} · chunk={chunk_size}", flush=True)

    prev_offload = getattr(model, "_offload_esmc", False)
    model._offload_esmc = bool(offload_esmc)
    if chunk_size != "default":
        model.set_chunk_size(chunk_size)

    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    w_gb = torch.cuda.memory_allocated() / 1024**3
    rec["vram_before_gb"] = round(w_gb, 3)

    try:
        spi = build_spi(chains)
        perm, order = token_order_audit(spi)
        rec["token_order"] = order
        if not order["is_permutation"]:
            rec["error"] = "token_order khong phai permutation hop le"
            return rec

        t = time.time()
        with warnings.catch_warnings(record=True) as w:
            warnings.simplefilter("always")
            result = builder.fold(model, spi, num_loops=num_loops,
                                  num_sampling_steps=num_sampling_steps,
                                  num_diffusion_samples=1, seed=seed, lm_dropout=lm_dropout,
                                  msa_max_depth=MSA_MAX_DEPTH,
                                  msa_column_mask_rate=MSA_COLUMN_MASK_RATE,
                                  complex_id=tag)
            rec["warnings"] = sorted({str(x.message)[:180] for x in w})[:8]
        rec["seconds"] = round(time.time() - t, 2)

        alloc, peak, reserved = vram()
        rec.update(vram_peak_gb=round(peak, 3), vram_reserved_peak_gb=round(reserved, 3),
                   vram_after_gb=round(alloc, 3),
                   activation_gb=round(peak - w_gb, 3),
                   vram_free_after_gb=round(torch.cuda.mem_get_info(0)[0] / 1024**3, 3))

        rec["has_pae"] = result.pae is not None
        rec["has_plddt"] = result.plddt is not None
        rec["ptm"] = None if result.ptm is None else round(float(result.ptm), 5)
        rec["iptm"] = None if result.iptm is None else round(float(result.iptm), 5)
        if result.plddt is not None:
            pl = result.plddt.detach().cpu().float().numpy().ravel()
            rec["plddt_mean_0to1"] = round(float(pl.mean()), 5)
            rec["plddt_scale_looks_0to1"] = bool(pl.max() <= 1.0)
        if result.pae is None:
            rec["error"] = "result.pae is None -> confidence head khong chay"
            return rec

        pae = result.pae.detach().cpu().float().numpy()
        rec["pae_shape"] = list(pae.shape)
        rec["pae_mean"] = round(float(pae.mean()), 6)
        if save_pae:
            p = f"{outdir}/{tag}.pae.npy"; np.save(p, pae); rec["pae_npy"] = p
        if result.pair_chains_iptm is not None:
            rec["pair_chains_iptm"] = np.round(
                result.pair_chains_iptm.detach().cpu().float().numpy(), 5).tolist()

        if do_ipsae:
            ip = run_ipsae(result, tag, outdir, perm=perm)
            rec["ipsae"] = ip
            if "ipsae_min" in ip:
                print(f"  ipSAE asym = {ip['asym']}")
                print(f"  ipSAE min = {ip['ipsae_min']:.4f}  max = {ip['ipsae_max']:.4f}")
            else:
                print(f"  ipSAE LỖI rc={ip['ipsae_returncode']} {ip.get('ipsae_stderr','')[:300]}")

        rec["ok"] = True
        print(f"  OK {rec['seconds']:.1f}s · peak {rec['vram_peak_gb']:.2f} GB "
              f"(activation {rec['activation_gb']:.2f} GB) · ptm={rec['ptm']} iptm={rec['iptm']} "
              f"· pLDDT {rec.get('plddt_mean_0to1')}")
        del result
    except torch.cuda.OutOfMemoryError as e:
        rec["error"] = "CUDA_OOM"
        rec["error_detail"] = str(e)[:400]
        rec["vram_peak_gb"] = round(torch.cuda.max_memory_allocated() / 1024**3, 3)
        print(f"  OOM ở {n_res} residue (peak {rec['vram_peak_gb']:.2f} GB)")
    except Exception as e:
        rec["error"] = f"{type(e).__name__}: {e}"
        rec["traceback"] = traceback.format_exc()[-2500:]
        print(f"  LỖI {rec['error']}")
        print(rec["traceback"][-1200:])
    finally:
        model._offload_esmc = prev_offload
        if chunk_size != "default":
            model.set_chunk_size(64)
        gc.collect(); torch.cuda.empty_cache()
    return rec

print("fold_once sẵn sàng (fail-soft: lỗi được ghi vào report, notebook chạy tiếp).")
budget("(cell 8 xong)")

## Cell 9 — CHỐT DTYPE bằng fold tối thiểu (rẻ nhất có thể)

80 residue, `num_loops=1`, `num_sampling_steps=4`, không MSA. Đây là bài kiểm tra
"forward có đi hết đường, có trả `result.pae` không" với giá gần như bằng 0.

**Thứ tự thử — rẻ trước:**

| # | Cấu hình | Reload? | Dự đoán (từ source) |
|---|---|---|---|
| 1 | `dtype=None, fp8, fused` | không (đã load) | ✅ đường tác giả thiết kế |
| 2 | `dtype=None, fp8, kernel=None` | không | nếu (1) chết vì Triton |
| 3 | `dtype=bfloat16, fp8` + cast 3 head về fp32 | có (~2 ph) | fix tại chỗ, weights nhẹ hơn 0.5 GB |
| 4 | `dtype=None, bf16` | có (~2 ph) | ESMC bf16, tốn +6.8 GB, chỉ khi fp8 hỏng |

**Không** thử `dtype=bfloat16` trần (bất kể `esmc_precision`) — source đã chứng minh nó
chết ở `distogram_head`, và micro-repro ở cell 6 vừa xác nhận. Đỡ 2 lần reload vô ích.

In [ ]:
# ============================== CELL 9: SETTLE DTYPE (minimal fold) ===========
budget("(bắt đầu cell 9)")

MIN_CHAINS = [("A", MIN_A, None), ("B", MIN_B, None)]
MIN_KW = dict(num_loops=1, num_sampling_steps=4, seed=0, do_ipsae=False, save_pae=False)

def cast_heads_fp32(m):
    """Fix tại chỗ cho model đã load bf16: 3 head sau trunk chạy NGOÀI autocast trên z fp32."""
    for name in ("distogram_head", "structure_head", "confidence_head", "inputs_embedder"):
        mod = getattr(m, name, None)
        if mod is not None:
            mod.float()
    return {n: str(next(getattr(m, n).parameters()).dtype)
            for n in ("distogram_head", "structure_head", "confidence_head")}

LADDER = [
    {"name": "fp32_trunk_esmc_fp8_fused", "dtype": None, "esmc": "fp8",
     "kernel": "fused", "reload": False, "cast_heads": False},
    {"name": "fp32_trunk_esmc_fp8_refkernel", "dtype": None, "esmc": "fp8",
     "kernel": None, "reload": False, "cast_heads": False},
    {"name": "bf16_trunk_esmc_fp8_heads_cast_fp32", "dtype": "bfloat16", "esmc": "fp8",
     "kernel": "fused", "reload": True, "cast_heads": True},
    {"name": "fp32_trunk_esmc_bf16_fused", "dtype": None, "esmc": "bf16",
     "kernel": "fused", "reload": True, "cast_heads": False},
]

WORKING = None
for i, cfg in enumerate(LADDER):
    if WORKING is not None:
        break
    if not have_budget(6, f"cấu hình {cfg['name']}"):
        break
    print(f"\n########## CẤU HÌNH {i+1}/{len(LADDER)}: {cfg['name']} ##########")
    entry = {"order": i + 1, **{k: str(v) for k, v in cfg.items()}}
    try:
        if cfg["reload"]:
            model, info = load_model(cfg["dtype"], cfg["esmc"], cfg["kernel"] or "fused")
            KERNEL_BACKEND = info["kernel_backend_effective"]
            if cfg["kernel"] is None:
                model.set_kernel_backend(None); KERNEL_BACKEND = None
            entry["load_info"] = info
        elif i > 0:
            # chỉ đổi kernel backend, khỏi reload
            model.set_kernel_backend(cfg["kernel"])
            KERNEL_BACKEND = cfg["kernel"]
        if cfg["cast_heads"]:
            entry["heads_after_cast"] = cast_heads_fp32(model)
            print(f"  cast 3 head về fp32: {entry['heads_after_cast']}")
    except Exception as e:
        entry["setup_error"] = f"{type(e).__name__}: {e}"
        print(f"  setup LỖI: {entry['setup_error']}")
        REPORT["configs_tried"].append(entry); save_partial(); continue

    r = fold_once(f"min_{cfg['name']}", MIN_CHAINS, **MIN_KW)
    entry.update(fold_ok=r["ok"], fold_seconds=r.get("seconds"),
                 fold_error=r.get("error"), has_pae=r.get("has_pae"),
                 pae_shape=r.get("pae_shape"), vram_peak_gb=r.get("vram_peak_gb"))
    REPORT["configs_tried"].append(entry); save_partial()
    if r["ok"] and r.get("has_pae"):
        WORKING = cfg
        print(f"\n  ===> CẤU HÌNH CHẠY ĐƯỢC: {cfg['name']}  ({r['seconds']:.1f}s, "
              f"pae {r['pae_shape']})")
    else:
        print(f"\n  ===> thất bại: {r.get('error')}")

if WORKING is None:
    REPORT["steps"]["dtype_settled"] = {"working_config": None,
                                        "tried": [c["name"] for c in LADDER]}
    save_partial()
    raise SystemExit(
        "DỪNG: không cấu hình nào fold xong end-to-end. Gửi out/02_partial.json cho Claude — "
        "trong đó có traceback của từng cấu hình (REPORT['configs_tried'] và REPORT['folds']).")

# đảm bảo kernel backend là cái của cấu hình thắng
try:
    model.set_kernel_backend(WORKING["kernel"]); KERNEL_BACKEND = WORKING["kernel"]
except Exception:
    model.set_kernel_backend(None); KERNEL_BACKEND = None

REPORT["steps"]["dtype_settled"] = {
    "working_config": WORKING["name"],
    "dtype": str(WORKING["dtype"]), "esmc_precision": WORKING["esmc"],
    "kernel_backend": str(KERNEL_BACKEND), "heads_cast_to_fp32": WORKING["cast_heads"],
    "n_configs_tried": len([c for c in REPORT["configs_tried"] if "fold_ok" in c]),
    "failed_configs": [{"name": c.get("name"), "error": c.get("fold_error") or c.get("setup_error")}
                       for c in REPORT["configs_tried"] if c.get("fold_ok") is False
                       or "setup_error" in c],
}
print(f"\nDÙNG TỪ ĐÂY: {WORKING['name']} · kernel={KERNEL_BACKEND}")
save_partial()
budget("(cell 9 xong — dtype đã chốt)")

## Cell 10 — ĐỐI CHỨNG DƯƠNG/ÂM + TÍNH TIỀN ĐỊNH

**Đây là cell quan trọng nhất của notebook.** Nếu ipSAE không tách được binder 7S5B
(đã validate thực nghiệm, ái lực picomolar) khỏi chính nó bị xáo trộn, thì thước đo hỏng
và mọi thứ phía sau vô nghĩa — kể cả design có tốt.

Ba fold, **đúng cấu hình cửa sàng** (219 + 55 = 274 residue, 20 loop, 200 step, 1 sample,
`seed=0`, MSA depth 921 chỉ cho target):

| tag | binder | mục đích |
|---|---|---|
| `gate_pos_r1` | 7S5B thật | điểm dương + VRAM/thời gian **thật** (diffusion giờ mới chạy) |
| `gate_pos_r2` | 7S5B thật, **y hệt** | **tính tiền định**: PAE có giống từng bit không? |
| `gate_neg` | xáo trộn | điểm âm |

Tiêu chí (từ `05-CODE-NOTES` §8): `ipsae_min(dương) ≥ 0.5` **và** gap `≥ 0.15`.

Về tiền định: `fold()` mặc định `lm_dropout=0.3` — ngẫu nhiên, fresh mask mỗi loop.
`seed=0` seed cả `_lm_dropout_context` lẫn noise diffusion (`processor.py`
`_seed_context`), nên **phải** tiền định... trừ khi có kernel CUDA không tiền định
(Triton atomics, `cdist`, SVD trong rigid-align). Cửa sàng ghi "1 seed / 1 sample" —
câu đó chỉ có nghĩa nếu model tiền định. Cell này trả lời bằng số.

In [ ]:
# ============================== CELL 10: POS/NEG CONTROL + DETERMINISM ========
budget("(bắt đầu cell 10)")

GATE_KW = dict(num_loops=GATE_NUM_LOOPS, num_sampling_steps=GATE_SAMPLING_STEPS,
               seed=0, lm_dropout=GATE_LM_DROPOUT, do_ipsae=True, save_pae=True)

print("Fold 1/3: gate_pos_r1 — binder 7S5B thật, MSA target depth "
      f"{MSA_IL7RA.depth}. Đây là lần đầu diffusion decoder thật sự chạy.")
POS1 = fold_once("gate_pos_r1", [("A", IL7RA, MSA_IL7RA), ("B", BINDER_POS, None)], **GATE_KW)

if POS1["ok"]:
    print(f"\n>>> Thời gian 1 fold cửa sàng ({POS1['n_residues']} aa, {GATE_NUM_LOOPS} loop, "
          f"{GATE_SAMPLING_STEPS} step) = {POS1['seconds']:.1f} s")
    print(f">>> 1000 fold như vậy ≈ {POS1['seconds']*1000/3600:.1f} GPU-giờ "
          f"≈ {POS1['seconds']*1000/3600*CU_PER_HOUR:.0f} CU  (ta còn ~88 CU)")
    print(f">>> VRAM peak THẬT = {POS1['vram_peak_gb']:.2f} GB "
          f"(weights {POS1['vram_before_gb']:.2f} + activation {POS1['activation_gb']:.2f})")

POS2 = None
if DO_DETERMINISM and POS1["ok"] and have_budget(8, "determinism (fold lặp lại)"):
    print("\nFold 2/3: gate_pos_r2 — Y HỆT fold trên, cùng seed=0. Kiểm tiền định.")
    POS2 = fold_once("gate_pos_r2", [("A", IL7RA, MSA_IL7RA), ("B", BINDER_POS, None)], **GATE_KW)

NEG = None
if POS1["ok"] and have_budget(8, "negative control"):
    print("\nFold 3/3: gate_neg — binder XÁO TRỘN (cùng thành phần aa, không còn cấu trúc).")
    NEG = fold_once("gate_neg", [("A", IL7RA, MSA_IL7RA), ("B", BINDER_NEG, None)], **GATE_KW)

# ---------------- tiền định ----------------
DET = {"tested": POS2 is not None and POS2["ok"]}
if DET["tested"]:
    a = np.load(POS1["pae_npy"]); b = np.load(POS2["pae_npy"])
    DET.update(
        pae_bitwise_identical=bool(np.array_equal(a, b)),
        pae_max_abs_diff=float(np.abs(a - b).max()),
        pae_mean_abs_diff=float(np.abs(a - b).mean()),
        ptm_identical=POS1["ptm"] == POS2["ptm"],
        iptm_identical=POS1["iptm"] == POS2["iptm"],
        ptm=[POS1["ptm"], POS2["ptm"]], iptm=[POS1["iptm"], POS2["iptm"]],
        plddt_mean=[POS1.get("plddt_mean_0to1"), POS2.get("plddt_mean_0to1")],
        ipsae_min=[POS1.get("ipsae", {}).get("ipsae_min"), POS2.get("ipsae", {}).get("ipsae_min")],
        seconds=[POS1["seconds"], POS2["seconds"]],
    )
    i1, i2 = DET["ipsae_min"]
    DET["ipsae_identical"] = (i1 is not None and i1 == i2)
    DET["ipsae_abs_diff"] = None if (i1 is None or i2 is None) else round(abs(i1 - i2), 6)
    print("\n" + "="*72)
    print("TÍNH TIỀN ĐỊNH (2 lần fold y hệt, seed=0)")
    print(f"  PAE giống từng bit        : {DET['pae_bitwise_identical']}")
    print(f"  PAE max |diff|            : {DET['pae_max_abs_diff']:.6g} Å")
    print(f"  ptm  {DET['ptm']}   iptm {DET['iptm']}")
    print(f"  ipSAE_min {DET['ipsae_min']}  |diff| = {DET['ipsae_abs_diff']}")
    if DET["pae_bitwise_identical"]:
        print("  => TIỀN ĐỊNH. 'seed=0, 1 sample' của cửa sàng có nghĩa. 1 fold là đủ.")
    else:
        print("  => KHÔNG tiền định! lm_dropout=0.3 và/hoặc kernel CUDA gây nhiễu.")
        print("     Hệ quả: 'cửa sàng 1 seed / 1 sample' là một lần RÚT THĂM. Muốn xếp hạng")
        print("     ổn định ta phải fold nhiều lần rồi lấy trung bình → đắt gấp n lần.")
        print("     Phải hỏi organizer: họ cố định seed nào, hay lm_dropout=0?")
else:
    print("\n(bỏ qua kiểm tiền định)")

# ---------------- đối chứng dương/âm ----------------
CTRL = {"pos_ok": POS1["ok"], "neg_ok": bool(NEG and NEG["ok"])}
p = POS1.get("ipsae", {}).get("ipsae_min") if POS1["ok"] else None
n = NEG.get("ipsae", {}).get("ipsae_min") if NEG and NEG["ok"] else None
CTRL.update(
    ipsae_min_pos=p, ipsae_min_neg=n,
    ipsae_asym_pos=POS1.get("ipsae", {}).get("asym") if POS1["ok"] else None,
    ipsae_asym_neg=NEG.get("ipsae", {}).get("asym") if NEG and NEG["ok"] else None,
    gap=None if (p is None or n is None) else round(p - n, 5),
    iptm_pos=POS1.get("iptm"), iptm_neg=NEG.get("iptm") if NEG else None,
    plddt_pos=POS1.get("plddt_mean_0to1"), plddt_neg=NEG.get("plddt_mean_0to1") if NEG else None,
    msa_depth_target=MSA_IL7RA.depth,
    msa_depth_realistic=bool(MSA_REPORT["IL7RA"]["msa_depth_realistic"]),
    threshold_pos=0.5, threshold_gap=0.15,
)
if p is not None and n is not None:
    CTRL["verdict"] = ("THUOC_DO_HOAT_DONG" if (p >= 0.5 and p - n >= 0.15)
                       else ("GAP_QUA_NHO" if p >= 0.5 else "BINDER_THAT_DIEM_THAP"))
else:
    CTRL["verdict"] = "KHONG_DU_DU_LIEU"

print("\n" + "="*72)
print("ĐỐI CHỨNG DƯƠNG/ÂM  —  KẾT QUẢ QUAN TRỌNG NHẤT")
print("="*72)
print(f"  binder 7S5B THẬT (validate, picomolar) : ipSAE_min = {p}")
print(f"  binder XÁO TRỘN (cùng aa composition)  : ipSAE_min = {n}")
print(f"  GAP                                    : {CTRL['gap']}   (cần ≥ 0.15)")
print(f"  iptm  dương/âm                         : {CTRL['iptm_pos']} / {CTRL['iptm_neg']}")
print(f"  pLDDT dương/âm (0–1)                   : {CTRL['plddt_pos']} / {CTRL['plddt_neg']}")
print(f"  MSA target depth                       : {CTRL['msa_depth_target']} "
      f"(realistic={CTRL['msa_depth_realistic']})")
print(f"  KẾT LUẬN                               : {CTRL['verdict']}")
if CTRL["verdict"] == "THUOC_DO_HOAT_DONG":
    print("\n  => Thước đo tách được binder thật khỏi sequence xáo. Đi tiếp sang generator được.")
elif CTRL["verdict"] == "BINDER_THAT_DIEM_THAP":
    print("\n  => BINDER THẬT ĐIỂM THẤP: setup của ta sai, không phải design sai.")
    print("     Nghi can theo thứ tự: (a) num_sampling_steps 200 vs 14, (b) lm_dropout 0.3 vs 0.25,")
    print("     (c) chiều/cutoff ipSAE, (d) MSA. KHÔNG được sinh design trước khi sửa xong.")
elif CTRL["verdict"] == "GAP_QUA_NHO":
    print("\n  => Gap quá nhỏ: không dùng ipSAE một mình để chọn 20 sequence được.")

REPORT["steps"]["determinism"] = DET
REPORT["steps"]["pos_neg_control"] = CTRL
save_partial()
budget("(cell 10 xong)")

## Cell 11 — ipSAE: có MSA vs KHÔNG MSA

Cửa sàng dùng MSA cho target. Nếu delta lớn, mọi điểm ta chấm **không có** MSA sẽ xếp
hạng sai candidate → phải luôn chấm kèm MSA, đắt hơn nhưng không có lựa chọn.
Cùng binder dương, cùng `seed=0`, chỉ khác `msa=None` cho chain target.

In [ ]:
# ============================== CELL 11: MSA ON vs OFF ========================
budget("(bắt đầu cell 11)")

MSAD = {"ran": False}
if DO_MSA_DELTA and POS1["ok"] and have_budget(8, "MSA on/off delta"):
    NOMSA = fold_once("gate_pos_nomsa", [("A", IL7RA, None), ("B", BINDER_POS, None)], **GATE_KW)
    a = POS1.get("ipsae", {}).get("ipsae_min")
    b = NOMSA.get("ipsae", {}).get("ipsae_min") if NOMSA["ok"] else None
    MSAD = {
        "ran": True,
        "ipsae_min_with_msa": a, "msa_depth": MSA_IL7RA.depth,
        "ipsae_min_no_msa": b,
        "delta": None if (a is None or b is None) else round(a - b, 5),
        "iptm_with_msa": POS1.get("iptm"), "iptm_no_msa": NOMSA.get("iptm"),
        "plddt_with_msa": POS1.get("plddt_mean_0to1"), "plddt_no_msa": NOMSA.get("plddt_mean_0to1"),
        "seconds_with_msa": POS1.get("seconds"), "seconds_no_msa": NOMSA.get("seconds"),
        "vram_peak_with_msa": POS1.get("vram_peak_gb"), "vram_peak_no_msa": NOMSA.get("vram_peak_gb"),
        "warnings_no_msa": NOMSA.get("warnings"),
    }
    print("\n" + "="*72)
    print(f"ipSAE_min  CÓ MSA (depth {MSA_IL7RA.depth}) = {a}")
    print(f"ipSAE_min  KHÔNG MSA (depth 1)       = {b}")
    print(f"DELTA                                = {MSAD['delta']}")
    print(f"thời gian  có/không MSA              = {MSAD['seconds_with_msa']}s / {MSAD['seconds_no_msa']}s")
    print(f"VRAM peak  có/không MSA              = {MSAD['vram_peak_with_msa']} / "
          f"{MSAD['vram_peak_no_msa']} GB")
    if MSAD["delta"] is not None:
        if abs(MSAD["delta"]) >= 0.1:
            print("  => MSA đổi điểm NHIỀU. Mọi lần chấm PHẢI có MSA, nếu không xếp hạng sai.")
        else:
            print("  => MSA đổi điểm ÍT. Có thể sàng rẻ không MSA rồi chỉ chấm lại top-N kèm MSA.")
else:
    print("(bỏ qua MSA delta)")

REPORT["steps"]["msa_delta"] = MSAD
save_partial()
budget("(cell 11 xong)")

## Cell 12 — TRẦN SỐ RESIDUE (giờ forward mới chạy hết)

Notebook 01 đo activation 0.875 GB @ 235 aa và 1.241 GB @ 280 aa (exponent 1.99 — bậc hai
rất sạch), nhưng **cả hai đều là sàn dưới vì diffusion decoder chưa chạy**. Cell này đo lại
với forward đi hết đường.

Leo thang TL1A (180 aa/protomer) + binder, **có MSA thật cho các chain TL1A** (depth 2037 →
đúng kiểu cửa sàng; MSA cũng tốn VRAM nên đây là con số trung thực):

| n protomer | binder | tổng | ý nghĩa |
|---|---|---|---|
| 1 | 100 | 280 | mốc đã biết của notebook 01 |
| 2 | 60 | 420 | vượt "500 aa" của maintainer chưa? |
| 3 | 0 | 540 | **trimer TL1A sinh học, chưa có binder** |
| 3 | 100 | 640 | **trimer + binder — mốc quyết định** |
| 3 | 235 | 775 | trần trên |

Tham số rẻ (`num_loops=4`, `num_sampling_steps=8`): VRAM gần như độc lập với số loop/step,
nên probe rẻ vẫn trả lời đúng câu hỏi trần. **Thời gian thì KHÔNG dùng được để ngoại suy.**

OOM được bắt riêng, rồi **thử lại cùng mốc đó** với `_offload_esmc=True` + `chunk_size=32`
(đẩy ESMC ≈6 GB sang CPU sau pha LM, `model.py:1063`) → ta có cả trần thường và trần
"max headroom".

In [ ]:
# ============================== CELL 12: RESIDUE CEILING ======================
budget("(bắt đầu cell 12)")

PROBE_KW = dict(num_loops=PROBE_NUM_LOOPS, num_sampling_steps=PROBE_SAMPLING_STEPS,
                seed=0, do_ipsae=False, save_pae=False)

def probe_chains(n_prot, binder, with_msa=True):
    """TL1A x n_prot (một ProteinInput, id=list -> MSA được broadcast cho cả n chain,
    xem prepare_input.py:1182) + binder không MSA."""
    ids = ["A", "B", "C"][:n_prot]
    ch = [(ids if n_prot > 1 else "A", TL1A, MSA_TL1A if with_msa else None)]
    if binder:
        ch.append(("Z", binder, None))
    return ch

def n_res_of(chains):
    return sum((len(c) if isinstance(c, list) else 1) * len(s) for c, s, _ in chains)

#            n_protomer, binder, tổng mong đợi
PROBE = [(1, BINDER_100, 280),
         (2, BINDER_60,  420),
         (3, "",         540),
         (3, BINDER_100, 640),
         (3, BINDER_235, 775)]
for n_prot, binder, expect in PROBE:
    assert n_res_of(probe_chains(n_prot, binder)) == expect, (n_prot, len(binder), expect)

def ceil_row(n_prot, binder, total, mode, msa_used, r):
    return {"n_protomer": n_prot, "binder_len": len(binder), "n_residues": total,
            "mode": mode, "msa_used": msa_used,
            "ok": r["ok"], "error": r.get("error"), "seconds": r.get("seconds"),
            "vram_peak_gb": r.get("vram_peak_gb"), "activation_gb": r.get("activation_gb"),
            "vram_before_gb": r.get("vram_before_gb"),
            "plddt_mean": r.get("plddt_mean_0to1"), "ptm": r.get("ptm")}

CEIL = []
max_ok, first_oom, best_point = 0, None, None
if DO_CEILING_PROBE:
    for n_prot, binder, total in PROBE:
        if not have_budget(5, f"probe {total} aa"):
            break
        ch = probe_chains(n_prot, binder)
        r = fold_once(f"probe_{total}", ch, **PROBE_KW)
        CEIL.append(ceil_row(n_prot, binder, total, "plain", True, r))
        REPORT["ceiling_probe"] = CEIL; save_partial()
        if r["ok"]:
            max_ok = max(max_ok, total); best_point = (n_prot, binder, total)
            continue
        if first_oom is None and r.get("error") == "CUDA_OOM":
            first_oom = total
        # thử lại CÙNG mốc với max headroom: ESMC sang CPU + chunk 32
        if not have_budget(5, f"probe {total} aa (offload+chunk32)"):
            break
        print(f"  -> thử lại {total} aa với _offload_esmc=True + chunk_size=32")
        r2 = fold_once(f"probe_{total}_offload", ch, offload_esmc=True, chunk_size=32, **PROBE_KW)
        CEIL.append(ceil_row(n_prot, binder, total, "offload_esmc+chunk32", True, r2))
        REPORT["ceiling_probe"] = CEIL; save_partial()
        if r2["ok"]:
            max_ok = max(max_ok, total); best_point = (n_prot, binder, total)
        else:
            break   # cả max headroom cũng chết -> đây là trần thật

    # giá VRAM của MSA: chạy lại ĐÚNG mốc lớn nhất đã OK, chỉ bỏ MSA
    if best_point and have_budget(5, "probe no-MSA (đo giá VRAM của MSA)"):
        n_prot, binder, total = best_point
        r = fold_once(f"probe_{total}_nomsa", probe_chains(n_prot, binder, with_msa=False),
                      **PROBE_KW)
        CEIL.append(ceil_row(n_prot, binder, total, "no_msa", False, r))
        REPORT["ceiling_probe"] = CEIL; save_partial()
else:
    print("(bỏ qua probe trần residue)")

print("\n" + "="*84)
print(f"{'residue':>8} {'mode':>22} {'ok':>5} {'peak GB':>9} {'activ GB':>9} {'giây':>7}  lỗi")
for c in CEIL:
    print(f"{c['n_residues']:>8} {c['mode']:>22} {str(c['ok']):>5} "
          f"{str(c.get('vram_peak_gb')):>9} {str(c.get('activation_gb')):>9} "
          f"{str(c.get('seconds')):>7}  {c.get('error') or ''}")
print("="*84)
print(f"Mốc lớn nhất chạy được: {max_ok} residue · OOM đầu tiên: {first_oom}")
print(f"VRAM tổng: {ENV['vram_total_gb']} GB")
if max_ok >= 640:
    print("=> TL1A FOLD ĐƯỢC CẢ TRIMER + binder trên L4. Hướng dẫn '24 GB ≈ 500 residue' của")
    print("   maintainer không áp dụng cho cấu hình ESMC fp8 (hướng dẫn đó hẳn nói về bf16/fp32).")
elif max_ok >= 540:
    print("=> Trimer TL1A fold được nhưng thêm binder thì chật. Cân nhắc offload + chunk 32.")
else:
    print("=> Không fold nổi trimer. Nhánh TL1A phải chấm theo monomer, hoặc thuê L40S/A100.")

REPORT["steps"]["ceiling"] = {"max_ok_residues": max_ok, "first_oom_residues": first_oom,
                              "vram_total_gb": ENV["vram_total_gb"],
                              "probe_num_loops": PROBE_NUM_LOOPS,
                              "probe_num_sampling_steps": PROBE_SAMPLING_STEPS,
                              "note": "tham so re -> VRAM dung duoc, THOI GIAN KHONG ngoai suy duoc"}
save_partial()
budget("(cell 12 xong)")

## Cell 13 — MONOMER vs TRIMER: câu hỏi chiến lược của nhánh TL1A

Cấu hình cửa sàng ghi `仅靶点序列使用 MSA` — "chỉ **sequence** target dùng MSA", số ít. Cộng
với việc phải chấm ~7000 sequence, cấu hình monomer rẻ là khả năng cao. **Nhưng ta không
biết.** Nếu hai cấu hình cho ipSAE khác nhau nhiều thì đây là rủi ro lớn nhất của nhánh
TL1A: ta có thể tối ưu sai mặt protein suốt cuộc thi.

Probe: **DcR3** (receptor TL1A tự nhiên, 176 aa, PDB 3K51) đấu với

- (a) **1 protomer** TL1A → 356 residue
- (b) **trimer** TL1A đầy đủ → 716 residue

Chọn DcR3 vì nó bám **rãnh giữa hai protomer kề nhau** — kiểu bám của cả họ receptor TNF.
Nếu monomer và trimer có khác nhau, nó sẽ lộ ra ở đây. Một binder bất kỳ không bám gì sẽ
cho ipSAE ≈ 0 ở **cả hai** cấu hình và delta ≈ 0 — kết luận "hai cấu hình như nhau" khi đó
là **giả**. Vì vậy ta đọc cả giá trị tuyệt đối, không chỉ delta.

**DcR3 không phải candidate để submit** — nó là receptor tự nhiên 176 aa, 20+ cysteine.

Hai nhánh dùng Y HỆT `num_loops`/`num_sampling_steps`/`seed`/`chunk_size`/`offload`, nếu
không delta vô nghĩa. Cả hai bật `_offload_esmc=True` (không đổi phép tính, chỉ chuyển
ESMC qua PCIe) để nhánh 716 aa có chỗ thở.

In [ ]:
# ============================== CELL 13: MONOMER vs TRIMER ====================
budget("(bắt đầu cell 13)")

MT = {"ran": False}
MT_KW = dict(num_loops=MT_NUM_LOOPS, num_sampling_steps=MT_SAMPLING_STEPS, seed=0,
             offload_esmc=True, do_ipsae=True, save_pae=False)

if DO_MONO_TRI and have_budget(12, "monomer vs trimer"):
    mono = fold_once("tl1a_mono_dcr3", [("A", TL1A, MSA_TL1A), ("Z", DCR3, None)], **MT_KW)
    tri  = fold_once("tl1a_tri_dcr3",
                     [(["A", "B", "C"], TL1A, MSA_TL1A), ("Z", DCR3, None)], **MT_KW)

    def z_pairs(rec):
        """ipSAE của các cặp có chain Z (binder) — chain target là A/B/C."""
        asym = rec.get("ipsae", {}).get("asym", {}) if rec.get("ok") else {}
        return {k: v for k, v in asym.items() if "Z" in k}

    mz, tz = z_pairs(mono), z_pairs(tri)
    m_best = max(mz.values()) if mz else None
    t_best = max(tz.values()) if tz else None
    MT = {
        "ran": True, "probe_binder": "DcR3 O95407 res30-205 (PDB 3K51) — PROBE, not a candidate",
        "settings_identical": True, "num_loops": MT_NUM_LOOPS,
        "num_sampling_steps": MT_SAMPLING_STEPS, "offload_esmc": True,
        "monomer": {"n_residues": mono["n_residues"], "ok": mono["ok"],
                    "error": mono.get("error"), "seconds": mono.get("seconds"),
                    "vram_peak_gb": mono.get("vram_peak_gb"), "iptm": mono.get("iptm"),
                    "ptm": mono.get("ptm"), "plddt_mean": mono.get("plddt_mean_0to1"),
                    "ipsae_asym_all": mono.get("ipsae", {}).get("asym"),
                    "ipsae_binder_pairs": mz, "ipsae_binder_best": m_best,
                    "pair_chains_iptm": mono.get("pair_chains_iptm")},
        "trimer": {"n_residues": tri["n_residues"], "ok": tri["ok"],
                   "error": tri.get("error"), "seconds": tri.get("seconds"),
                   "vram_peak_gb": tri.get("vram_peak_gb"), "iptm": tri.get("iptm"),
                   "ptm": tri.get("ptm"), "plddt_mean": tri.get("plddt_mean_0to1"),
                   "ipsae_asym_all": tri.get("ipsae", {}).get("asym"),
                   "ipsae_binder_pairs": tz, "ipsae_binder_best": t_best,
                   "pair_chains_iptm": tri.get("pair_chains_iptm")},
        "delta_best": None if (m_best is None or t_best is None) else round(t_best - m_best, 5),
    }
    d = MT["delta_best"]
    print("\n" + "="*72)
    print("MONOMER vs TRIMER (cùng DcR3, cùng tham số)")
    print(f"  (a) TL1A x1 + DcR3  = {MT['monomer']['n_residues']} aa  "
          f"ipSAE(binder) = {m_best}  iptm = {mono.get('iptm')}  all = {mz}")
    print(f"  (b) TL1A x3 + DcR3  = {MT['trimer']['n_residues']} aa  "
          f"ipSAE(binder) = {t_best}  iptm = {tri.get('iptm')}  all = {tz}")
    print(f"  DELTA (trimer − monomer) = {d}")
    if d is None:
        print("  => thiếu dữ liệu (một nhánh lỗi) — xem 'error' trong report.")
        MT["verdict"] = "KHONG_DU_DU_LIEU"
    elif (m_best or 0) < 0.1 and (t_best or 0) < 0.1:
        print("  => CẢ HAI gần 0 → phép so sánh KHÔNG kết luận được gì. DcR3 không fold ra phức")
        print("     hợp lý (nhiều cysteine, không khai báo disulfide). Phải lặp lại với binder khác.")
        MT["verdict"] = "KHONG_KET_LUAN_CA_HAI_GAN_0"
    elif abs(d) >= 0.1:
        print("  => HAI CẤU HÌNH KHÁC NHAU RÕ. Đây là rủi ro lớn nhất của nhánh TL1A:")
        print("     fold monomer rồi tối ưu sẽ nhắm sai mặt protein. PHẢI HỎI ORGANIZER:")
        print("     '靶点序列' của TL1A là 1 protomer hay trimer sinh học?")
        MT["verdict"] = "KHAC_NHAU_PHAI_HOI_ORGANIZER"
    else:
        print("  => Hai cấu hình cho điểm gần nhau → câu hỏi này không chặn ta. Dùng monomer (rẻ hơn).")
        MT["verdict"] = "GIONG_NHAU_DUNG_MONOMER"
else:
    print("(bỏ qua monomer vs trimer)")

REPORT["steps"]["monomer_vs_trimer"] = MT
save_partial()
budget("(cell 13 xong)")

## Cell 14 — BÁO CÁO

In block `PASTE CHO CLAUDE` **và** ghi `out/02_report.json` lên Drive.
Chọn cách nào tiện: copy block text, **hoặc** tải file JSON gửi vào chat.

Chỗ `<<DIEN TAY>>`: số **compute unit còn lại** (Colab, menu trên phải → "Compute units").
Điền trước và sau khi chạy để tôi biết notebook này thật sự tốn bao nhiêu.

In [ ]:
# ============================== CELL 14: REPORT ===============================
POS1 = next((r for r in REPORT["folds"] if r["tag"] == "gate_pos_r1"), {}) or {}
NEGR = next((r for r in REPORT["folds"] if r["tag"] == "gate_neg"), {}) or {}
S = REPORT["steps"]

REPORT["finished_utc"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
REPORT["elapsed_min"] = round(elapsed_min(), 2)
REPORT["cu_estimate"] = round(cu_burned(), 3)

ctrl = S.get("pos_neg_control", {})
det  = S.get("determinism", {})
ceil_ = S.get("ceiling", {})
mt    = S.get("monomer_vs_trimer", {})
msad  = S.get("msa_delta", {})
dts   = S.get("dtype_settled", {})

REPORT["headline"] = {
    "dtype_root_cause": ("from_pretrained(dtype=bfloat16) cast distogram_head/structure_head/"
                         "confidence_head sang bf16, nhung forward() chay chung NGOAI autocast "
                         "tren z fp32 (model.py:1159 z=z.float(), autocast dong o 1160) => "
                         "mat1 Float x mat2 BFloat16 tai distogram_head (model.py:1181)"),
    "root_cause_reproduced": S.get("root_cause_repro", {}).get("reproduces_notebook01_error"),
    "working_config": dts.get("working_config"),
    "failed_configs": dts.get("failed_configs"),
    "vram_weights_gb": S.get("load_model", {}).get("vram_weights_gb"),
    "diffusion_decoder_ran_in_nb01": False,
    "gate_fold_seconds_true": POS1.get("seconds"),
    "gate_fold_vram_peak_gb_true": POS1.get("vram_peak_gb"),
    "gate_fold_activation_gb_true": POS1.get("activation_gb"),
    "gate_fold_residues": POS1.get("n_residues"),
    "ipsae_pos_7S5B": ctrl.get("ipsae_min_pos"),
    "ipsae_neg_scrambled": ctrl.get("ipsae_min_neg"),
    "ipsae_gap": ctrl.get("gap"),
    "control_verdict": ctrl.get("verdict"),
    "deterministic_pae_bitwise": det.get("pae_bitwise_identical"),
    "deterministic_ipsae": det.get("ipsae_identical"),
    "msa_source_il7ra": S.get("msa", {}).get("IL7RA", {}).get("source"),
    "msa_depth_il7ra": S.get("msa", {}).get("IL7RA", {}).get("msa_depth_loaded"),
    "msa_source_tl1a": S.get("msa", {}).get("TL1A", {}).get("source"),
    "msa_depth_tl1a": S.get("msa", {}).get("TL1A", {}).get("msa_depth_loaded"),
    "msa_all_realistic": S.get("msa_all_realistic"),
    "ipsae_delta_msa_vs_nomsa": msad.get("delta"),
    "max_residues_ok": ceil_.get("max_ok_residues"),
    "first_oom_residues": ceil_.get("first_oom_residues"),
    "tl1a_trimer_feasible": (None if ceil_.get("max_ok_residues") is None
                             else ceil_.get("max_ok_residues", 0) >= 640),
    "monomer_vs_trimer_ipsae": [mt.get("monomer", {}).get("ipsae_binder_best"),
                                mt.get("trimer", {}).get("ipsae_binder_best")],
    "monomer_vs_trimer_delta": mt.get("delta_best"),
    "monomer_vs_trimer_verdict": mt.get("verdict"),
}

if POS1.get("seconds"):
    s = POS1["seconds"]
    REPORT["headline"]["cost_model"] = {
        "seconds_per_gate_fold": s,
        "gpu_hours_per_1000_folds": round(s * 1000 / 3600, 2),
        "cu_per_1000_folds": round(s * 1000 / 3600 * CU_PER_HOUR, 1),
        "cu_per_100_folds": round(s * 100 / 3600 * CU_PER_HOUR, 2),
        "note": "L4 Colab Pro ~4.8 CU/gio. Con ~88 CU => so fold toi da o cau hinh cua sang.",
        "max_gate_folds_in_88cu": int(88 / (s / 3600 * CU_PER_HOUR)) if s else None,
    }

out = f"{D_OUT}/02_report.json"
tmp = out + ".tmp"
with open(tmp, "w") as f:
    json.dump(REPORT, f, indent=2, default=str)
os.replace(tmp, out)

print("=" * 40)
print("PASTE CHO CLAUDE — BẮT ĐẦU")
print("=" * 40)
print(json.dumps(REPORT, indent=2, default=str))
print("=" * 40)
print("PASTE CHO CLAUDE — KẾT THÚC")
print("=" * 40)

print(f"\nĐã ghi: {out}")
print(f"Thư mục run (PDB/JSON/PAE từng fold): {D_RUNS}")
print(f"a3m đã dùng: {D_MSA}")
print(f"\nTổng {REPORT['elapsed_min']:.1f} phút · ước {REPORT['cu_estimate']:.2f} CU")

print("\n" + "=" * 72)
print("TÓM TẮT")
print("=" * 72)
h = REPORT["headline"]
rows = [
    ("Cấu hình dtype chạy được",       h["working_config"]),
    ("Tái hiện được nguyên nhân gốc",  h["root_cause_reproduced"]),
    ("VRAM weights",                   f"{h['vram_weights_gb']} GB"),
    ("1 fold cửa sàng (274 aa)",       f"{h['gate_fold_seconds_true']} s · peak "
                                       f"{h['gate_fold_vram_peak_gb_true']} GB"),
    ("ipSAE binder THẬT (7S5B)",       h["ipsae_pos_7S5B"]),
    ("ipSAE binder XÁO TRỘN",          h["ipsae_neg_scrambled"]),
    ("GAP (cần ≥ 0.15)",               h["ipsae_gap"]),
    ("Kết luận đối chứng",             h["control_verdict"]),
    ("Tiền định (PAE từng bit)",       h["deterministic_pae_bitwise"]),
    ("MSA IL-7Rα",                     f"{h['msa_source_il7ra']} depth {h['msa_depth_il7ra']}"),
    ("MSA TL1A",                       f"{h['msa_source_tl1a']} depth {h['msa_depth_tl1a']}"),
    ("ipSAE delta MSA vs không MSA",   h["ipsae_delta_msa_vs_nomsa"]),
    ("Trần residue",                   f"{h['max_residues_ok']} aa (OOM đầu: {h['first_oom_residues']})"),
    ("TL1A trimer khả thi trên L4",    h["tl1a_trimer_feasible"]),
    ("monomer vs trimer ipSAE",        f"{h['monomer_vs_trimer_ipsae']} delta "
                                       f"{h['monomer_vs_trimer_delta']}"),
    ("Kết luận monomer/trimer",        h["monomer_vs_trimer_verdict"]),
]
for k, v in rows:
    print(f"  {k:34s} : {v}")
if "cost_model" in h:
    c = h["cost_model"]
    print(f"\n  CHI PHÍ: {c['seconds_per_gate_fold']} s/fold → {c['cu_per_1000_folds']} CU cho 1000 fold")
    print(f"           với ~88 CU còn lại: tối đa ~{c['max_gate_folds_in_88cu']} fold cấu hình cửa sàng")

print("\n" + "=" * 72)
print("BA CÂU PHẢI HỎI ORGANIZER (chưa code nào trả lời được)")
print("=" * 72)
print("  1. num_sampling_steps: config.json ghi structure_head.inference_num_steps = 14,")
print("     nhưng builder.fold() default = 200. Dùng số nào?")
print("  2. lm_dropout: fold() default 0.3 ('paper folding-eval value'), config.json ghi")
print("     lm_encoder.lm_dropout = 0.25. Và nó NGẪU NHIÊN →", end=" ")
print(f"tiền định ở seed cố định = {h['deterministic_pae_bitwise']}")
print("  3. TL1A '靶点序列': 1 protomer hay trimer? →", h["monomer_vs_trimer_verdict"])
print("\n  Điền tay: compute unit còn lại trước/sau khi chạy (menu trên phải của Colab).")